In [1]:
!pip install scikit-learn

In [2]:
import sys
!{sys.executable} -m pip install wfdb scikit-learn

In [3]:
# =====================================================================================
# Project: PTB-XL Metadata Audit  -  v0.6 LOCK GATE: final-run and fold-10 analysis protocol
# Code version: v0.6
# Date: 2026-10-02
# CPU ONLY. torch and wfdb are NOT imported. No model is trained or evaluated.
#
# Purpose:
#   STEP 0  verify the four write-once preregistration documents (self_sha256 + file sha256),
#           their parent chain, selected_lr, the v0.3 / v0.4 / v0.5 manifests, input CSV
#           hashes, the v0.4 signal cache, and the v0.1.1 / v0.1.2 label files (bytes only,
#           never parsed); detect an existing final_run_protocol.json (never overwritten)
#   A  environment regression check (v0.5 Part A verbatim: v0.3 LogisticRegression B1/B3
#      fold-9 macro-AUROC must reproduce within 1e-5) + bootstrap-implementation regression
#      check (the locked reference bootstrap code must reproduce v0.3_bootstrap_deltas.csv)
#   B  LOCKED QUANTITIES: categorical encoder, age scaler and per-lead normalization refit on
#      folds 1-9; meta dims and head input widths; OneCycle step counts and the weight-decay
#      shrink factor of untrained one-hot columns; fold-10 METADATA counts (n records,
#      n patients, patient disjointness, __UNKNOWN__ counts); fold-10 bootstrap index matrix
#      and permutation index matrix (built from patient_id / record count only) and their hashes
#   C  write final_run_protocol.json ONCE (write_once_json, exclusive create, self_sha256)
#
# FOLD-10 ACCESS (hard rules):
#   - no fold-10 signal file is opened; wfdb is not imported
#   - pd.read_csv parses ptbxl_database.csv once (all rows, all columns); IMMEDIATELY after,
#     only ecg_id (index), patient_id, device, site, nurse of the fold-10 rows are copied into
#     fold10_meta (strat_fold is used only as the row filter), and df_all is restricted to
#     strat_fold <= 9, so no fold-10 label / age / sex value remains in any variable
#   - no label column (scp_codes or anything derived) and no age/sex statistic of fold 10 is
#     ever computed; only counts and the patient_id-derived index matrices are stored
#
# REUSE (verbatim from v0.5): helpers, verify_self_sha256, canonical hashing, write_once_json,
# to_level_string, encode_categorical, age sentinel handling, label derivation, v0.5 Part A,
# two-pass float64 lead normalization (v0.4). The only change in Part A: VERSIONS has no torch
# entry (CPU-only stage) and the per-class LR probabilities are kept for the bootstrap check.
#
# RE-RUN BEHAVIOUR: OUT_DIR files (env regression, preview, manifest) are regenerated on every
# run (not write-once). final_run_protocol.json is NEVER overwritten: if it exists, every field
# this code would write (except "timestamp") is compared with the file; equal -> "already
# locked", nothing is written to PREREG_DIR; different -> abort.
#
# Outputs:
#   OUT_DIR = <PROJECT_ROOT>\outputs\v0.6
#     v0.6_env_regression.csv
#     v0.6_locked_quantities_preview.json   (written BEFORE final_run_protocol.json)
#     v0.6_manifest.json                    (hashes of everything, incl. the new document)
#   PREREG_DIR = <PROJECT_ROOT>\preregistration
#     final_run_protocol.json               (write-once)
# =====================================================================================

import os, sys, json, ast, hashlib, platform, time, math, re, gc, ntpath
from datetime import datetime
import numpy as np
import pandas as pd
import scipy
from scipy.stats import rankdata
import sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

_GUARDED_MODULES = ("torch", "wfdb")
_MODULES_AT_START = {m for m in _GUARDED_MODULES if m in sys.modules}

pd.set_option("display.max_rows", 300)
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)

# ------------------------------- configuration ---------------------------------------
PROJECT_ROOT = r"D:\ptbxl-metadata-audit"
DATA_ROOT  = os.path.join(PROJECT_ROOT, "data", "ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3")
OUT_DIR    = os.path.join(PROJECT_ROOT, "outputs", "v0.6")
V011_DIR   = os.path.join(PROJECT_ROOT, "outputs", "v0.1.1")
V012_DIR   = os.path.join(PROJECT_ROOT, "outputs", "v0.1.2")
V03_DIR    = os.path.join(PROJECT_ROOT, "outputs", "v0.3")
V04_DIR    = os.path.join(PROJECT_ROOT, "outputs", "v0.4")
V05_DIR    = os.path.join(PROJECT_ROOT, "outputs", "v0.5")
PREREG_DIR = os.path.join(PROJECT_ROOT, "preregistration")

SEAL_PATH             = os.path.join(PREREG_DIR, "pre_registration.json")
DEV_PROTOCOL_PATH     = os.path.join(PREREG_DIR, "dev_protocol.json")
TRAINING_DETAILS_PATH = os.path.join(PREREG_DIR, "v0.5_training_details.json")
LR_SELECTION_PATH     = os.path.join(PREREG_DIR, "lr_selection.json")
FINAL_PROTOCOL_PATH   = os.path.join(PREREG_DIR, "final_run_protocol.json")

V011_LABELS_CSV   = os.path.join(V011_DIR, "v0.1.1_labels.csv")
V011_MANIFEST     = os.path.join(V011_DIR, "v0.1.1_manifest.json")
V012_LABELS_CSV   = os.path.join(V012_DIR, "v0.1.2_labels.csv")
V012_MANIFEST     = os.path.join(V012_DIR, "v0.1.2_manifest.json")
V03_MANIFEST      = os.path.join(V03_DIR, "v0.3_manifest.json")
V03_BOOT_CSV      = os.path.join(V03_DIR, "v0.3_bootstrap_deltas.csv")
V04_MANIFEST      = os.path.join(V04_DIR, "v0.4_manifest.json")
SIGNALS_NPY       = os.path.join(V04_DIR, "v0.4_signals_folds1to9_100hz.npy")
SIGNALS_INDEX     = os.path.join(V04_DIR, "v0.4_signals_index.csv")
V04_LEAD_NORM_CSV = os.path.join(V04_DIR, "v0.4_lead_norm_params.csv")
V05_MANIFEST      = os.path.join(V05_DIR, "v0.5_manifest.json")

CODE_VERSION = "v0.6"
PREFIX = CODE_VERSION + "_"
SUPERCLASSES = ["NORM", "MI", "STTC", "CD", "HYP"]
ACQ_COLS = ["device", "site", "nurse"]

# --- write-once documents: self_sha256 (canonical JSON) and file sha256 (bytes on disk) ---
EXPECTED_SELF_SHA256 = {
    "pre_registration": "5183f54569f3358a5339b16a2c3c3be49f5c7798eabe6b0bb25946119c2ba469",
    "dev_protocol":     "c967ac38c8cc9382bf41fdd3785c8bbd730db196e349846b8052d856bbac5fc6",
    "training_details": "c12c33a2f649984fb1b70a46445823c46b57d1b024d87655e16661789f04b46f",
    "lr_selection":     "c70cbfa65e0ab75321dfef9c64714a64705c60a967685d79a878a1d40450749d",
}
EXPECTED_FILE_SHA256 = {          # pre_registration: read from v0.3_manifest.json (asserted)
    "dev_protocol":     "babf101c1d69b322daef6907175fdc4c17b5a53755a68f13a998eefa0148dc2d",
    "training_details": "6480c191aeaed9ff1e6b1da5caa0c1e46908d453d30106a32beb7cf83678be11",
    "lr_selection":     "358d783b37bba6fcf61237a43817c1fd8594fe192f8ecdfca2c25b871a21415b",
}
DOC_PATHS = {"pre_registration": SEAL_PATH, "dev_protocol": DEV_PROTOCOL_PATH,
             "training_details": TRAINING_DETAILS_PATH, "lr_selection": LR_SELECTION_PATH}
EXPECTED_SELECTED_LR = {"resnet1d_wang": 0.01, "inception1d": 0.01}

# --- manifests of earlier stages (file sha256 as printed by those stages) ---
EXPECTED_V03_MANIFEST_SHA256 = "c7e1bec283d11a3d4cd94a3f2509640651e8069d26424cfd14b5d6b975053e35"
EXPECTED_V04_MANIFEST_SHA256 = "3781f90c4cf5a85ea612efabb1fd81cc61b77cbeb38a54cc644c10c66afe1d9b"
EXPECTED_V05_MANIFEST_SHA256 = "59968bfb50d8c919de0e8004ef878a44a115560de438b7e74582877495e95f87"
EXPECTED_V012_MANIFEST_SHA256 = "cfcb071691c25fe7aa5db6b6cb71bce4848dba7ae541de2ca335d51458572c0d"
EXPECTED_V011_LABELS_SHA256 = "236dc3d8d608ca96554b3a974834b00b522a28746e37b3f514d878facf3e437b"

EXPECTED_INPUT_HASHES = {
    "ptbxl_database.csv": "7600de9c1b27d181d850b3c6038a35d7c3ddb6bb33b702e3a20252a6859d216b",
    "scp_statements.csv": "ad05b0b1fcae83bb1230755ad9cfc7c96f303feddc08a4a9ad5bdc9ca63bac8f",
}
EXPECTED_SIGNALS_NPY_SHA256 = "17558f9d09b39b990d7f3a5ca2ea10e1527980288a49a6828c89b7c3df61745a"
EXPECTED_SIGNALS_INDEX_SHA256 = "06b49ecdafc6a74c9bdbcfa0177bb357eb845ece187b40e3a777496a8cf72203"
EXPECTED_LEAD_NORM_CSV_SHA256 = "07fb6d6dc8f0f1e2379fdd45b6462f1102518621eb242ba585fb8600acff35aa"

# --- data constants (v0.1.1 / v0.2.1 / v0.4 / v0.5) ---
EXPECTED_N_SOURCE_RECORDS = 21799
EXPECTED_N_RECORDS_FOLDS1TO9 = 19601
EXPECTED_N_RECORDS_FOLDS1TO8 = 17418
EXPECTED_N_FOLD10_RECORDS = 2198
EXPECTED_N_FOLD10_PATIENTS = 1904
EXPECTED_N_AGE_SENTINEL_FOLDS1TO9 = 259
EXPECTED_FOLD9_POSITIVES = {"NORM": 955, "MI": 540, "STTC": 528, "CD": 495, "HYP": 268}
EXPECTED_ONEHOT_COUNTS_FOLDS1TO8 = {"device": 13, "site": 52, "nurse": 14}
V05_AGE_SCALER_FOLDS1TO8 = {"mean": 59.7333, "scale": 16.8750}      # printed with 4 decimals
AGE_SCALER_TOL = 5e-5

# Part A: v0.3 LogisticRegression seed-42 fold-9 macro-AUROC (6 decimals)
V03_LR_MACRO_AUROC = {"B1_physio": 0.688969, "B3_physio_plus_acquisition": 0.697165}
ENV_REGRESSION_TOL = 1e-5
V03_BOOT_MAX_ABS_DIFF_TOL = 1e-4   # per-draw delta; a different index stream gives ~1e-2 diffs
V03_LR_CI95 = (-0.0001, 0.0168)    # v0.3 LogisticRegression paired-delta CIs (printed with 4 decimals)
V03_LR_CI90 = (0.0013, 0.0153)

# Cross-check (reviewer): one-hot columns that are all-zero in folds 1-9 training. device has no
# missing values in PTB-XL (v0.1.1), so device=__MISSING__ is all-zero; __UNKNOWN__ is all-zero
# by construction. Any other result -> STOP before anything is written.
EXPECTED_ALLZERO_TRAIN_COLS_FOLDS1TO9 = ["device=__MISSING__", "device=__UNKNOWN__",
                                         "nurse=__UNKNOWN__", "site=__UNKNOWN__"]
EXPECTED_FOLD10_UNTRAINED_COUNT = 0   # fold-10 records hitting any all-zero training column

# constants that dev_protocol.json defines (asserted equal to the file; the FILE values are used)
EXPECTED_EPOCHS = 50
EXPECTED_LR_GRID = [1e-3, 3e-3, 1e-2]
EXPECTED_CROP = 250
EXPECTED_EVAL_STRIDE = 125
EXPECTED_WINDOW_STARTS = [0, 125, 250, 375, 500, 625, 750]
EXPECTED_WEIGHT_DECAY = 1e-2
EXPECTED_BATCH_SIZE = {"resnet1d_wang": 128, "inception1d": 128}

SEED = 42
N_SAMPLES = 1000
N_LEADS = 12
EMBEDDING_DIM = 256
CHUNK = 1000
ONECYCLE = {"pct_start": 0.3, "div_factor": 25.0, "final_div_factor": 1e4, "anneal_strategy": "cos"}
EXPECTED_N_BATCHES_FINAL = 154
EXPECTED_TOTAL_STEPS_FINAL = 7700
EXPECTED_LAST_BATCH_FINAL = 17
EXPECTED_N_BATCHES_DEV = 137
EXPECTED_TOTAL_STEPS_DEV = 6850

# fold-10 analysis constants (sealed: N_BOOT=2000, BOOT_SEED=7; reviewer: N_PERM=20, PERM_SEED=42)
N_BOOT = 2000
BOOT_SEED = 7
N_PERM = 20
PERM_SEED = 42
SESOI_H3 = 0.01
NAN_DRAW_FLAG_FRACTION = 0.01
FINAL_SEEDS = [42, 123, 2024]
ARCHS = ["resnet1d_wang", "inception1d"]
ARMS = ["signal_only", "signal_plus_physio", "signal_plus_all"]

_root_abs = os.path.abspath(DATA_ROOT)
for _p in (OUT_DIR, PREREG_DIR):
    assert not os.path.abspath(_p).lower().startswith(_root_abs.lower()), \
        "output path is inside the dataset folder - aborting: " + _p

RUN_STARTED = datetime.now().astimezone().isoformat()
OUTPUT_FILES = []
WARNINGS = []


def sha256_of(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


def sha256_bytes(b):
    return hashlib.sha256(b).hexdigest()


def save_csv(obj, name, index=True):
    p = os.path.join(OUT_DIR, name)
    obj.to_csv(p, index=index, encoding="utf-8-sig")
    OUTPUT_FILES.append(p)
    print("[saved] " + p)
    return p


def hdr(t):
    print("\n" + "=" * 100)
    print(t)
    print("=" * 100)


def warn(msg):
    WARNINGS.append(msg)
    print("[WARN] " + msg)


def note(msg):
    print("[NOTE] " + msg)


def _json_default(o):
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, np.floating):
        return float(o)
    if isinstance(o, np.bool_):
        return bool(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    return str(o)


def to_level_string(series, missing="__MISSING__"):
    """Human-readable, zero-padded categorical level for numeric columns (e.g.
    '007' for site) so sorting and printed feature names stay legible."""
    if pd.api.types.is_numeric_dtype(series):
        nn = series.dropna()
        integral = bool(len(nn) == 0 or np.all(np.isclose(nn.values, np.round(nn.values))))
        if integral:
            width = max(3, len(str(int(nn.max()))) if len(nn) else 3)
            return series.apply(lambda v: missing if pd.isna(v) else str(int(v)).zfill(width))
    return series.apply(lambda v: missing if pd.isna(v) else str(v).strip())


def encode_categorical(level_series, train_mask, col_name):
    """One-hot encode a pre-stringified categorical column. The category list is
    fixed from the training rows only, always includes an explicit '__MISSING__'
    level, and always includes a catch-all '__UNKNOWN__' level for anything seen
    only outside training (e.g. only in fold 9). Both splits therefore produce the
    exact same set of columns."""
    train_vals = level_series[train_mask]
    train_categories = sorted(set(train_vals.unique().tolist()) | {"__MISSING__"})
    known = set(train_categories)
    remapped = level_series.where(level_series.isin(known), other="__UNKNOWN__")
    categories = train_categories + ["__UNKNOWN__"]
    cat = pd.Categorical(remapped, categories=categories)
    dummies = pd.get_dummies(cat)
    dummies.columns = ["{}={}".format(col_name, c) for c in dummies.columns]
    dummies.index = level_series.index
    return dummies, categories


def remap_with_categories(level_series, categories):
    """Same remapping rule as encode_categorical, with a FIXED category list (whose last
    element is '__UNKNOWN__'): any level not among the training categories -> '__UNKNOWN__'."""
    assert categories[-1] == "__UNKNOWN__"
    known = set(categories[:-1])
    return level_series.where(level_series.isin(known), other="__UNKNOWN__")


def canonical_json_bytes(d):
    """Canonical serialization used for self_sha256 (see SELF_SHA256_PROCEDURE)."""
    return json.dumps(d, sort_keys=True, ensure_ascii=False, separators=(",", ":"),
                      default=_json_default).encode("utf-8")


SELF_SHA256_PROCEDURE = (
    "Load the JSON file; remove the key 'self_sha256' (the key must be absent, not empty); "
    "serialize the remaining dict with json.dumps(d, sort_keys=True, ensure_ascii=False, "
    "separators=(',', ':')); compute SHA-256 of the UTF-8 bytes; the hex digest must equal "
    "the stored 'self_sha256'. (At write time numpy scalars were converted to plain JSON "
    "types first, so the loaded file hashes identically.)"
)


def verify_self_sha256(path):
    """Return (stored, recomputed, loaded_dict) for a self-hashed JSON document."""
    with open(path, "r", encoding="utf-8") as f:
        d = json.load(f)
    stored = d.pop("self_sha256", None)
    recomputed = hashlib.sha256(canonical_json_bytes(d)).hexdigest()
    d["self_sha256"] = stored
    return stored, recomputed, d


def fast_auc(y, s):
    """Binary AUROC via the Mann-Whitney U statistic (average-rank tie handling).
    Used instead of sklearn.metrics.roc_auc_score for speed inside the bootstrap
    loop; both compute the identical quantity."""
    y = np.asarray(y)
    n1 = int(y.sum())
    n0 = len(y) - n1
    if n1 == 0 or n0 == 0:
        return float("nan")
    ranks = rankdata(np.asarray(s))
    return float((ranks[y == 1].sum() - n1 * (n1 + 1) / 2.0) / (n1 * n0))


def write_once_json(path, doc):
    """Normalize to plain JSON types, add self_sha256, exclusive-create, verify from disk."""
    doc = json.loads(json.dumps(doc, ensure_ascii=False, default=_json_default))
    assert "self_sha256" not in doc
    sha = hashlib.sha256(canonical_json_bytes(doc)).hexdigest()
    doc["self_sha256"] = sha
    with open(path, "x", encoding="utf-8") as f:          # refuses to overwrite
        json.dump(doc, f, ensure_ascii=False, indent=2)
    s_, r_, _ = verify_self_sha256(path)
    assert s_ == r_ == sha, "self_sha256 verification FAILED for " + path
    return sha


def plain(o):
    return json.loads(json.dumps(o, ensure_ascii=False, default=_json_default))


def manifest_entry(manifest, basename):
    """Return the unique 'outputs' entry of a project manifest whose file basename matches
    (works for full Windows paths and bare names)."""
    hits = [o for o in manifest["outputs"] if ntpath.basename(o["file"]) == basename]
    assert len(hits) == 1, "expected exactly one manifest entry for {}, found {}".format(basename, len(hits))
    return hits[0]


def load_verified_manifest(path, expected_sha, label):
    assert os.path.isfile(path), label + " not found: " + path + " - aborting (fix the path constant)."
    got = sha256_of(path)
    assert got == expected_sha, "{} file sha256 {} != expected {} - aborting.".format(label, got, expected_sha)
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f), got


# =====================================================================================
# REFERENCE IMPLEMENTATION (locked verbatim inside final_run_protocol.json; v0.9 must exec
# this exact string, asserting its sha256, and use these functions for every fold-10 number)
# =====================================================================================
REFERENCE_CODE = r'''
import numpy as np
from scipy.stats import rankdata


def fast_auc(y, s):
    """Binary AUROC via the Mann-Whitney U statistic (average-rank tie handling).
    Used instead of sklearn.metrics.roc_auc_score for speed inside the bootstrap
    loop; both compute the identical quantity."""
    y = np.asarray(y)
    n1 = int(y.sum())
    n0 = len(y) - n1
    if n1 == 0 or n0 == 0:
        return float("nan")
    ranks = rankdata(np.asarray(s))
    return float((ranks[y == 1].sum() - n1 * (n1 + 1) / 2.0) / (n1 * n0))


def patient_rows_in_file_order(patient_ids):
    """patient_ids: 1-D integer array, one entry per evaluation record, in file order
    (ptbxl_database.csv row order). Returns (patients, patient_row_list): patients =
    np.unique(patient_ids) (sorted); patient_row_list[i] = row positions of patients[i]."""
    patient_ids = np.asarray(patient_ids, dtype=np.int64)
    patients = np.unique(patient_ids)
    patient_row_list = [np.where(patient_ids == p)[0] for p in patients]
    return patients, patient_row_list


def build_boot_idx(n_patients, n_boot, boot_seed):
    """N_BOOT sequential calls rng.integers(0, n_patients, size=n_patients) on ONE
    Generator(boot_seed); row b = call b (exactly as v0.3). A single call with
    size=(n_boot, n_patients) is NOT used."""
    rng = np.random.default_rng(boot_seed)
    boot_idx = np.empty((n_boot, n_patients), dtype=np.int64)
    for b in range(n_boot):
        boot_idx[b] = rng.integers(0, n_patients, size=n_patients)
    return boot_idx


def build_perm_idx(n_records, n_perm, perm_seed):
    """n_perm sequential calls rng.permutation(n_records) on ONE Generator(perm_seed)."""
    rng = np.random.default_rng(perm_seed)
    perm_idx = np.empty((n_perm, n_records), dtype=np.int64)
    for r in range(n_perm):
        perm_idx[r] = rng.permutation(n_records)
    return perm_idx


def boot_rows(patient_row_list, idx_row):
    return np.concatenate([patient_row_list[i] for i in idx_row])


def per_class_auroc(Y, P, rows=None):
    """Y: (n, 5) 0/1 labels, P: (n, 5) probabilities, columns NORM, MI, STTC, CD, HYP."""
    if rows is not None:
        Y = Y[rows]
        P = P[rows]
    return np.array([fast_auc(Y[:, k], P[:, k]) for k in range(Y.shape[1])], dtype=np.float64)


def macro_auroc_strict(Y, P, rows=None):
    """Mean of the 5 per-class AUROCs; NaN if ANY class AUROC is undefined (no positives
    or no negatives) - the whole draw is then excluded (NOT a per-class nan-skip)."""
    per = per_class_auroc(Y, P, rows)
    if np.isnan(per).any():
        return float("nan")
    return float(per.mean())


def paired_bootstrap(Y, P_a, P_b, patient_row_list, boot_idx):
    """Fully paired: both arms are evaluated on the SAME drawn rows. Returns
    (macro_delta (n_boot,), per_class_delta (n_boot, 5)); delta = arm_B - arm_A."""
    n_boot = boot_idx.shape[0]
    macro_delta = np.empty(n_boot, dtype=np.float64)
    class_delta = np.empty((n_boot, Y.shape[1]), dtype=np.float64)
    for b in range(n_boot):
        rows = boot_rows(patient_row_list, boot_idx[b])
        pa = per_class_auroc(Y, P_a, rows)
        pb = per_class_auroc(Y, P_b, rows)
        class_delta[b] = pb - pa
        if np.isnan(pa).any() or np.isnan(pb).any():
            macro_delta[b] = float("nan")
        else:
            macro_delta[b] = float(pb.mean()) - float(pa.mean())
    return macro_delta, class_delta


def summarize_deltas(point_delta, deltas, nan_flag_fraction=0.01):
    """Percentile CIs over the valid (non-NaN) draws; np.percentile method='linear';
    no rounding before any comparison."""
    deltas = np.asarray(deltas, dtype=np.float64)
    valid = deltas[~np.isnan(deltas)]
    n_nan = int(len(deltas) - len(valid))
    lo95, hi95 = np.percentile(valid, [2.5, 97.5], method="linear")
    lo90, hi90 = np.percentile(valid, [5.0, 95.0], method="linear")
    return {"point_delta": float(point_delta), "lo95": float(lo95), "hi95": float(hi95),
            "lo90": float(lo90), "hi90": float(hi90), "n_valid": int(len(valid)), "n_nan": n_nan,
            "nan_fraction": n_nan / len(deltas), "nan_flag": bool(n_nan / len(deltas) > nan_flag_fraction)}


def h3_verdict(lo95, hi95, lo90, hi90, sesoi=0.01):
    """Four-state H3 verdict resolving the overlap of sealed rules a and b."""
    a_holds = bool(lo95 > 0.0 or hi95 < 0.0)
    b_holds = bool(lo90 >= -sesoi and hi90 <= sesoi)
    material = bool(lo95 > sesoi or hi95 < -sesoi)
    sign = "+" if lo95 > 0.0 else ("-" if hi95 < 0.0 else None)
    if a_holds and b_holds:
        verdict = "statistically detectable, below the materiality threshold"
    elif a_holds and material:
        verdict = "acquisition metadata materially changes performance (sign: {})".format(sign)
    elif a_holds:
        verdict = ("acquisition metadata changes performance (sign: {}); magnitude relative to "
                   "SESOI undetermined".format(sign))
    elif b_holds:
        verdict = "no material change"
    else:
        verdict = "inconclusive"
    sealed_rules_holding = [r for r, h in (("a", a_holds), ("b", b_holds)) if h] or ["c"]
    return {"a_holds": a_holds, "b_holds": b_holds, "material": material, "sign": sign,
            "verdict": verdict, "sealed_rules_holding": sealed_rules_holding}
'''
REFERENCE_CODE_SHA256 = sha256_bytes(REFERENCE_CODE.encode("utf-8"))
REF = {}
exec(REFERENCE_CODE, REF)


# =====================================================================================
# STEP 0 - INTEGRITY (before anything is written)
# =====================================================================================
hdr("STEP 0 - INTEGRITY")

if _MODULES_AT_START:
    warn("module(s) {} were already imported in this kernel by an earlier cell; v0.6 does not use "
         "them (restart the kernel for a clean lock run).".format(sorted(_MODULES_AT_START)))

DOCS, DOC_SELF, DOC_FILE = {}, {}, {}
for key, path in DOC_PATHS.items():
    assert os.path.isfile(path), key + " not found: " + path + " - aborting."
    _s, _r, _d = verify_self_sha256(path)
    assert _s == _r == EXPECTED_SELF_SHA256[key], \
        "{}: self_sha256 stored {} / recomputed {} / expected {} - aborting.".format(key, _s, _r, EXPECTED_SELF_SHA256[key])
    DOCS[key], DOC_SELF[key], DOC_FILE[key] = _d, _s, sha256_of(path)
    print("{:18s} self_sha256 {} -> PASS".format(key, _s))
SEAL, DEV, TD, LR_SEL = DOCS["pre_registration"], DOCS["dev_protocol"], DOCS["training_details"], DOCS["lr_selection"]
assert SEAL.get("sealed") is True

for key, exp in EXPECTED_FILE_SHA256.items():
    assert DOC_FILE[key] == exp, "{}: file sha256 {} != {} - aborting.".format(key, DOC_FILE[key], exp)
    print("{:18s} file sha256 {} -> PASS".format(key, DOC_FILE[key]))

V03_MAN, V03_MAN_SHA = load_verified_manifest(V03_MANIFEST, EXPECTED_V03_MANIFEST_SHA256, "v0.3_manifest.json")
_seal_entry = manifest_entry(V03_MAN, "pre_registration.json")
assert DOC_FILE["pre_registration"] == _seal_entry["sha256"], \
    "pre_registration.json file sha256 {} != v0.3 manifest record {} - aborting.".format(
        DOC_FILE["pre_registration"], _seal_entry["sha256"])
assert V03_MAN["pre_registration_self_sha256"] == EXPECTED_SELF_SHA256["pre_registration"]
print("{:18s} file sha256 {} == v0.3_manifest.json record -> PASS".format("pre_registration", DOC_FILE["pre_registration"]))

# --- parent chain ---
assert DEV["parent_seal_self_sha256"] == DOC_SELF["pre_registration"]
assert TD["parent_seal_self_sha256"] == DOC_SELF["pre_registration"]
assert TD["parent_dev_protocol_self_sha256"] == DOC_SELF["dev_protocol"]
assert LR_SEL["parent_hashes"] == {"pre_registration": DOC_SELF["pre_registration"],
                                   "dev_protocol": DOC_SELF["dev_protocol"],
                                   "training_details": DOC_SELF["training_details"]}
SELECTED_LR = {k: float(v) for k, v in LR_SEL["selected_lr"].items()}
assert SELECTED_LR == EXPECTED_SELECTED_LR, "selected_lr {} != {} - aborting.".format(SELECTED_LR, EXPECTED_SELECTED_LR)
print("parent chain dev_protocol -> seal; training_details -> seal + dev_protocol; lr_selection -> "
      "seal + dev_protocol + training_details: PASS | selected_lr {}: PASS".format(SELECTED_LR))

# --- recipe values loaded FROM dev_protocol.json (verbatim v0.5), asserted equal to the constants ---
_rec = DEV["training_recipe"]
EPOCHS = int(_rec["epochs"])
LR_GRID = [float(v) for v in DEV["tuning_grid"]["learning_rate"]]
WEIGHT_DECAY = float(_rec["weight_decay"])
BATCH_SIZE = {k: int(v) for k, v in _rec["batch_size"].items()}
_m = re.search(r"random (\d+)-sample crops", _rec["train_input"])
assert _m, "cannot parse the crop length from dev_protocol train_input"
CROP = int(_m.group(1))
_m = re.search(r"sliding windows of (\d+) samples, stride (\d+) \(window starts \[([0-9, ]+)\]\)", _rec["evaluation"])
assert _m, "cannot parse the evaluation windows from dev_protocol evaluation"
EVAL_WINDOW = int(_m.group(1))
EVAL_STRIDE = int(_m.group(2))
WINDOW_STARTS = [int(v) for v in _m.group(3).split(",")]
AGGREGATION = "mean" if "probability = mean over windows" in _rec["evaluation"] else None
assert "MEAN over sliding windows" in DEV["evaluation_aggregation_note"]
assert _rec["optimizer"] == "AdamW" and _rec["lr_schedule"] == "one-cycle"
assert EPOCHS == EXPECTED_EPOCHS and LR_GRID == EXPECTED_LR_GRID and WEIGHT_DECAY == EXPECTED_WEIGHT_DECAY
assert BATCH_SIZE == EXPECTED_BATCH_SIZE and CROP == EVAL_WINDOW == EXPECTED_CROP
assert EVAL_STRIDE == EXPECTED_EVAL_STRIDE and WINDOW_STARTS == EXPECTED_WINDOW_STARTS and AGGREGATION == "mean"
# training details: values this protocol relies on
_tdd = TD["details"]
assert _tdd["epochs"] == EPOCHS and _tdd["batch_size"] == BATCH_SIZE
assert _tdd["optimizer"]["weight_decay"] == WEIGHT_DECAY and _tdd["optimizer"]["name"] == "torch.optim.AdamW"
assert _tdd["lr_scheduler"]["pct_start"] == ONECYCLE["pct_start"]
assert _tdd["lr_scheduler"]["div_factor"] == ONECYCLE["div_factor"]
assert _tdd["lr_scheduler"]["final_div_factor"] == ONECYCLE["final_div_factor"]
assert _tdd["lr_scheduler"]["anneal_strategy"] == ONECYCLE["anneal_strategy"]
assert _tdd["lr_scheduler"]["three_phase"] is False
print("recipe (dev_protocol + training_details): epochs={} lr_grid={} wd={} bs={} crop={} windows={} "
      "aggregation={} OneCycle={} -> PASS".format(EPOCHS, LR_GRID, WEIGHT_DECAY, BATCH_SIZE, CROP,
                                                 WINDOW_STARTS, AGGREGATION, ONECYCLE))

# --- input CSVs ---
DB_CSV = os.path.join(DATA_ROOT, "ptbxl_database.csv")
SCP_CSV = os.path.join(DATA_ROOT, "scp_statements.csv")
for p in (DB_CSV, SCP_CSV):
    assert os.path.isfile(p), "file not found: " + p
INPUT_HASHES = {os.path.basename(p): sha256_of(p) for p in (DB_CSV, SCP_CSV)}
assert INPUT_HASHES == EXPECTED_INPUT_HASHES, "input CSV hashes differ from v0.4/v0.5 - aborting."
assert INPUT_HASHES == SEAL["input_files_sha256"] == DEV["input_files_sha256"]
print("input CSV hashes      : equal to v0.4/v0.5 and to the seal -> PASS")

# --- v0.4 manifest, signal cache, lead-norm csv ---
V04_MAN, V04_MAN_SHA = load_verified_manifest(V04_MANIFEST, EXPECTED_V04_MANIFEST_SHA256, "v0.4_manifest.json")
CACHE_HASHES = {}
for path, expected in [(SIGNALS_NPY, EXPECTED_SIGNALS_NPY_SHA256), (SIGNALS_INDEX, EXPECTED_SIGNALS_INDEX_SHA256),
                       (V04_LEAD_NORM_CSV, EXPECTED_LEAD_NORM_CSV_SHA256)]:
    assert os.path.isfile(path), "v0.4 file missing: " + path
    got = sha256_of(path)
    assert got == expected == manifest_entry(V04_MAN, os.path.basename(path))["sha256"], \
        "v0.4 file hash mismatch for " + path + " - aborting."
    CACHE_HASHES[os.path.basename(path)] = got
    print("  {}  {}  -> PASS (== v0.4 manifest)".format(got, os.path.basename(path)))
assert manifest_entry(V04_MAN, "dev_protocol.json")["sha256"] == DOC_FILE["dev_protocol"]

# --- v0.5 manifest (anchors the three runs v0.7 reuses) ---
V05_MAN, V05_MAN_SHA = load_verified_manifest(V05_MANIFEST, EXPECTED_V05_MANIFEST_SHA256, "v0.5_manifest.json")
assert manifest_entry(V05_MAN, "v0.5_training_details.json")["sha256"] == DOC_FILE["training_details"]
assert manifest_entry(V05_MAN, "lr_selection.json")["sha256"] == DOC_FILE["lr_selection"]
V07_REUSED_RUNS = [("resnet1d_wang", "signal_plus_physio"), ("inception1d", "signal_plus_physio"),
                   ("resnet1d_wang", "signal_only")]
V05_REUSED_FILES = {}
for arch, arm in V07_REUSED_RUNS:
    base = "v0.5_{}_{}_seed42_lr0.01".format(arch, arm)
    for suffix in ("_fold9_probs.csv", "_train_loss.csv", "_state_dict.pt"):
        V05_REUSED_FILES[base + suffix] = manifest_entry(V05_MAN, base + suffix)["sha256"]
print("v0.5_manifest.json    : {} -> PASS ({} reused-run files recorded for v0.7)".format(V05_MAN_SHA, len(V05_REUSED_FILES)))

# --- v0.1.1 / v0.1.2 label files: sha256 of BYTES ONLY (never parsed) ---
assert os.path.isfile(V011_LABELS_CSV), "v0.1.1 labels file not found: " + V011_LABELS_CSV + " (fix the path constant)"
assert os.path.isfile(V011_MANIFEST), "v0.1.1 manifest not found: " + V011_MANIFEST + " (fix the path constant)"
V011_LABELS_SHA = sha256_of(V011_LABELS_CSV)
assert V011_LABELS_SHA == EXPECTED_V011_LABELS_SHA256, "v0.1.1_labels.csv sha256 differs - aborting."
with open(V011_MANIFEST, "r", encoding="utf-8") as f:
    assert EXPECTED_V011_LABELS_SHA256 in f.read(), "v0.1.1 manifest does not record the labels sha256 - aborting."
V011_MAN_SHA = sha256_of(V011_MANIFEST)
assert os.path.isfile(V012_LABELS_CSV), "v0.1.2 labels file not found: " + V012_LABELS_CSV + " (fix the path constant)"
V012_LABELS_SHA = sha256_of(V012_LABELS_CSV)
V012_MAN_SHA = sha256_of(V012_MANIFEST) if os.path.isfile(V012_MANIFEST) else None
assert V012_MAN_SHA == EXPECTED_V012_MANIFEST_SHA256, \
    "v0.1.2_manifest.json sha256 {} != {} - aborting.".format(V012_MAN_SHA, EXPECTED_V012_MANIFEST_SHA256)
with open(V012_MANIFEST, "r", encoding="utf-8") as f:
    V012_MAN_RECORDS_LABELS = V012_LABELS_SHA in f.read()
print("v0.1.1_labels.csv     : {} == expected and recorded in v0.1.1 manifest -> PASS (bytes only)".format(V011_LABELS_SHA))
print("v0.1.2_labels.csv     : {} (bytes only, file not parsed) | recorded in v0.1.2 manifest: {}".format(
    V012_LABELS_SHA, "YES" if V012_MAN_RECORDS_LABELS else "NO"))
if not V012_MAN_RECORDS_LABELS:
    warn("v0.1.2_manifest.json does not contain the sha256 of v0.1.2_labels.csv (format may differ); "
         "the file sha256 is recorded in the protocol as computed.")
print("v0.1.2_manifest.json  : {} -> PASS".format(V012_MAN_SHA))

# --- existing final_run_protocol.json? (never overwritten) ---
FINAL_EXISTS = os.path.exists(FINAL_PROTOCOL_PATH)
FINAL_ON_DISK = None
if FINAL_EXISTS:
    _s, _r, FINAL_ON_DISK = verify_self_sha256(FINAL_PROTOCOL_PATH)
    assert _s == _r, "final_run_protocol.json BROKEN (stored != recomputed self_sha256) - aborting."
    print("final_run_protocol.json EXISTS (self_sha256 {}, verified) -> it will be compared field by field; "
          "nothing will be written to PREREG_DIR.".format(_s))
else:
    print("final_run_protocol.json not present -> Part C will write it once (exclusive create).")

os.makedirs(OUT_DIR, exist_ok=True)


# =====================================================================================
# DATA: fold-10 metadata extraction, then folds 1-9 only
# =====================================================================================
hdr("DATA, LABELS, METADATA (folds 1-9) + FOLD-10 METADATA COLUMNS")

df_all = pd.read_csv(DB_CSV, index_col="ecg_id")
N_SOURCE = len(df_all)
assert N_SOURCE == EXPECTED_N_SOURCE_RECORDS
fold10_meta = df_all.loc[df_all["strat_fold"] == 10, ["patient_id", "device", "site", "nurse"]].copy()
df_all = df_all[df_all["strat_fold"] <= 9]                 # no fold-10 column remains in df_all
gc.collect()
FOLD10_COLUMNS_KEPT = ["ecg_id (index)", "patient_id", "device", "site", "nurse"]
FOLD10_ACCESS_LINE = (
    "fold-10 access: pd.read_csv parsed ptbxl_database.csv once (all {} rows, all columns); immediately "
    "afterwards only {} of the {} fold-10 rows were copied into fold10_meta (strat_fold used only as the "
    "row filter) and df_all was restricted to strat_fold <= 9. No fold-10 label, age or sex value is held "
    "in any variable or used in any computation.".format(N_SOURCE, FOLD10_COLUMNS_KEPT, len(fold10_meta)))
print(FOLD10_ACCESS_LINE)
assert list(fold10_meta.columns) == ["patient_id", "device", "site", "nurse"]
assert set(df_all["strat_fold"].unique()) == set(range(1, 10)), "df_all is not exactly folds 1-9 - aborting."

df9 = df_all.copy()
mask_train = df9["strat_fold"] <= 8          # folds 1-8 (development fit, regression checks)
mask_eval = df9["strat_fold"] == 9
mask_final = df9["strat_fold"] <= 9          # folds 1-9 (final-run fit) - all True
assert (df9["strat_fold"] == 10).sum() == 0, "fold 10 leaked into the working set - aborting."
assert bool(mask_final.all())
train_patients = set(df9.loc[mask_train, "patient_id"])
eval_patients = set(df9.loc[mask_eval, "patient_id"])
assert train_patients.isdisjoint(eval_patients), "patient-level leakage folds 1-8 vs fold 9 - aborting."
current_index = pd.DataFrame({
    "ecg_id": df9.index.to_numpy(),
    "strat_fold": df9["strat_fold"].to_numpy(),
    "patient_id": df9["patient_id"].to_numpy(),
})
N_REC = len(current_index)
assert N_REC == EXPECTED_N_RECORDS_FOLDS1TO9
train_rows = np.where(mask_train.to_numpy())[0]
eval_rows = np.where(mask_eval.to_numpy())[0]
all_rows = np.arange(N_REC)
assert len(train_rows) == EXPECTED_N_RECORDS_FOLDS1TO8
print("records: folds 1-8 = {} | fold 9 = {} | folds 1-9 = {} | fold-10 metadata rows kept = {}".format(
    len(train_rows), len(eval_rows), N_REC, len(fold10_meta)))

# --- label derivation (verbatim from v0.2.1), folds 1-9 only ---
scp = pd.read_csv(SCP_CSV, index_col=0)
diag = scp[scp["diagnostic"].fillna(0) == 1]
code2class = {k: v for k, v in diag["diagnostic_class"].items() if isinstance(v, str)}
parsed = df9["scp_codes"].apply(ast.literal_eval)
rec_classes = parsed.apply(lambda d: {code2class[k] for k in d if k in code2class})
lab = pd.DataFrame({c: rec_classes.apply(lambda s, c=c: int(c in s)) for c in SUPERCLASSES},
                   index=df9.index)
assert (current_index["ecg_id"].to_numpy() == lab.index.to_numpy()).all()
assert len(set(lab.index) & set(fold10_meta.index)) == 0, "a fold-10 ecg_id is in the labels index - aborting."
fold9_pos = {c: int(lab.loc[mask_eval, c].sum()) for c in SUPERCLASSES}
assert fold9_pos == EXPECTED_FOLD9_POSITIVES, "fold-9 label counts differ from v0.2.1 - aborting."
print("labels: folds 1-9 only (no fold-10 ecg_id in the index: asserted) | fold-9 positives", fold9_pos, "-> PASS")
Y_EVAL = {sc: lab.loc[mask_eval, sc].to_numpy() for sc in SUPERCLASSES}

# --- metadata preprocessing (verbatim from v0.5) ---
age_raw = df9["age"]
age_sentinel_mask = (age_raw == 300)
n_age_sentinel = int(age_sentinel_mask.sum())
assert n_age_sentinel == EXPECTED_N_AGE_SENTINEL_FOLDS1TO9
age_clean = age_raw.where(~age_sentinel_mask, np.nan)
age_censored = age_sentinel_mask.astype(float)
age_imputed = age_clean.fillna(90.0)
sex = df9["sex"].astype(float)
print("age: sentinel 300 affects", n_age_sentinel, "records (folds 1-9); flagged via age_censored, imputed with 90.")

LEVELS = {col: to_level_string(df9[col]) for col in ACQ_COLS}
B1_cols = ["age_imputed", "age_censored", "sex"]

# --- folds 1-8 encoder (development fit; Part A + regression check) ---
ENC18, CATS18 = {}, {}
for col in ACQ_COLS:
    ENC18[col], CATS18[col] = encode_categorical(LEVELS[col], mask_train, col)
    assert ENC18[col].shape[1] == EXPECTED_ONEHOT_COUNTS_FOLDS1TO8[col], \
        "{} folds 1-8 one-hot width {} != {} - aborting.".format(col, ENC18[col].shape[1], EXPECTED_ONEHOT_COUNTS_FOLDS1TO8[col])
device_dummies, site_dummies, nurse_dummies = ENC18["device"], ENC18["site"], ENC18["nurse"]
B2_cols = list(device_dummies.columns) + list(site_dummies.columns) + list(nurse_dummies.columns)
B3_cols = B1_cols + B2_cols


# =====================================================================================
# PART A - ENVIRONMENT REGRESSION CHECK (CPU) - v0.5 Part A verbatim
# =====================================================================================
hdr("PART A - ENVIRONMENT REGRESSION CHECK (v0.3 LogisticRegression, CPU)")

VERSIONS = {"python": sys.version.split()[0], "numpy": np.__version__, "pandas": pd.__version__,
            "scipy": scipy.__version__, "sklearn": sklearn.__version__, "os": platform.platform()}
print("versions:", VERSIONS)

FEATURES_V021 = pd.concat([age_imputed.rename("age_imputed"), age_censored.rename("age_censored"),
                           sex.rename("sex"), device_dummies, site_dummies, nurse_dummies], axis=1)
y_train_full = lab.loc[mask_train]
regression_rows = []
PARTA_PROBS = {}                     # block -> (n_fold9, 5) LR probabilities (kept for the bootstrap check)
for block_name, cols in [("B1_physio", B1_cols), ("B3_physio_plus_acquisition", B3_cols)]:
    Xtr_raw = FEATURES_V021.loc[mask_train, cols].to_numpy(dtype=float)
    Xev_raw = FEATURES_V021.loc[mask_eval, cols].to_numpy(dtype=float)
    block_cols = cols
    continuous_cols = [c for c in block_cols if c in ("age_imputed", "recording_year")]
    cont_idx = [block_cols.index(c) for c in continuous_cols]
    rest_idx = [i for i in range(len(block_cols)) if i not in cont_idx]
    scaler = StandardScaler().fit(Xtr_raw[:, cont_idx])
    Xtr_lr = np.empty_like(Xtr_raw)
    Xev_lr = np.empty_like(Xev_raw)
    Xtr_lr[:, cont_idx] = scaler.transform(Xtr_raw[:, cont_idx])
    Xev_lr[:, cont_idx] = scaler.transform(Xev_raw[:, cont_idx])
    if rest_idx:
        Xtr_lr[:, rest_idx] = Xtr_raw[:, rest_idx]
        Xev_lr[:, rest_idx] = Xev_raw[:, rest_idx]
    per_class = []
    probs_block = np.zeros((len(eval_rows), len(SUPERCLASSES)), dtype=np.float64)
    for k, sc in enumerate(SUPERCLASSES):
        model_lr = LogisticRegression(max_iter=2000, C=1.0, solver="lbfgs", random_state=SEED)
        model_lr.fit(Xtr_lr, y_train_full[sc].to_numpy())
        pos_col = list(model_lr.classes_).index(1)
        proba = model_lr.predict_proba(Xev_lr)[:, pos_col]
        probs_block[:, k] = proba
        per_class.append(fast_auc(Y_EVAL[sc], proba))
    PARTA_PROBS[block_name] = probs_block
    macro = float(np.nanmean(per_class))
    ref = V03_LR_MACRO_AUROC[block_name]
    ok = abs(macro - ref) <= ENV_REGRESSION_TOL
    regression_rows.append({"block": block_name, "learner": "LogisticRegression", "seed": SEED,
                            "macro_auroc_now": macro, "macro_auroc_v0.3": ref, "abs_diff": abs(macro - ref),
                            "tolerance": ENV_REGRESSION_TOL, "pass": ok, **VERSIONS})
    print("  {:28s} now={:.6f}  v0.3={:.6f}  |diff|={:.2e}  -> {}".format(
        block_name, macro, ref, abs(macro - ref), "PASS" if ok else "FAIL"))
env_reg_df = pd.DataFrame(regression_rows)
save_csv(env_reg_df, PREFIX + "env_regression.csv", index=False)
assert all(r["pass"] for r in regression_rows), \
    "ENVIRONMENT REGRESSION FAILED: v0.3 LogisticRegression results do not reproduce - aborting."
print("environment regression check: PASS")

# --- bootstrap-implementation regression: the LOCKED reference code must reproduce v0.3 ---
print("\n--- bootstrap implementation check: locked reference code vs v0.3_bootstrap_deltas.csv (fold 9) ---")
assert os.path.isfile(V03_BOOT_CSV), "v0.3_bootstrap_deltas.csv not found: " + V03_BOOT_CSV
_v03_boot_sha = sha256_of(V03_BOOT_CSV)
assert _v03_boot_sha == manifest_entry(V03_MAN, "v0.3_bootstrap_deltas.csv")["sha256"], \
    "v0.3_bootstrap_deltas.csv differs from its v0.3 manifest record - aborting."
_v03_boot = pd.read_csv(V03_BOOT_CSV, encoding="utf-8-sig")
assert len(_v03_boot) == N_BOOT and (_v03_boot["replicate"].to_numpy() == np.arange(N_BOOT)).all()
_pid9 = df9.loc[mask_eval, "patient_id"].to_numpy()
assert np.all(_pid9 == np.round(_pid9)), "non-integral patient_id in fold 9"
_Y9 = lab.loc[mask_eval, SUPERCLASSES].to_numpy(dtype=np.int64)
t0 = time.time()
_pat9, _rows9 = REF["patient_rows_in_file_order"](_pid9.astype(np.int64))
_boot9 = REF["build_boot_idx"](len(_pat9), N_BOOT, BOOT_SEED)
_d9, _ = REF["paired_bootstrap"](_Y9, PARTA_PROBS["B1_physio"], PARTA_PROBS["B3_physio_plus_acquisition"], _rows9, _boot9)
_point9 = REF["macro_auroc_strict"](_Y9, PARTA_PROBS["B3_physio_plus_acquisition"]) - \
          REF["macro_auroc_strict"](_Y9, PARTA_PROBS["B1_physio"])
_sum9 = REF["summarize_deltas"](_point9, _d9, NAN_DRAW_FLAG_FRACTION)
_maxdiff9 = float(np.nanmax(np.abs(_d9 - _v03_boot["delta_LR"].to_numpy())))
BOOT_IMPL_CHECK = {"fold9_patients": int(len(_pat9)), "n_nan_draws": _sum9["n_nan"],
                   "max_abs_diff_per_draw_vs_v0.3": _maxdiff9, "tolerance": V03_BOOT_MAX_ABS_DIFF_TOL,
                   "point_delta": _sum9["point_delta"], "ci95": [_sum9["lo95"], _sum9["hi95"]],
                   "ci90": [_sum9["lo90"], _sum9["hi90"]],
                   "v0.3_ci95_rounded": list(V03_LR_CI95), "v0.3_ci90_rounded": list(V03_LR_CI90)}
print("  {} draws in {:.1f}s | fold-9 patients {} | NaN draws {} | max |delta - v0.3 delta| = {:.2e} (tol {})".format(
    N_BOOT, time.time() - t0, len(_pat9), _sum9["n_nan"], _maxdiff9, V03_BOOT_MAX_ABS_DIFF_TOL))
print("  point {:+.4f} | 95%CI [{:+.4f},{:+.4f}] (v0.3 {}) | 90%CI [{:+.4f},{:+.4f}] (v0.3 {})".format(
    _sum9["point_delta"], _sum9["lo95"], _sum9["hi95"], list(V03_LR_CI95), _sum9["lo90"], _sum9["hi90"], list(V03_LR_CI90)))
assert _sum9["n_nan"] == 0 and _maxdiff9 <= V03_BOOT_MAX_ABS_DIFF_TOL, \
    "locked reference bootstrap does NOT reproduce v0.3 - aborting."
for got, exp in [(_sum9["lo95"], V03_LR_CI95[0]), (_sum9["hi95"], V03_LR_CI95[1]),
                 (_sum9["lo90"], V03_LR_CI90[0]), (_sum9["hi90"], V03_LR_CI90[1])]:
    assert abs(got - exp) <= 5e-5 + 1e-9, "CI bound {:.6f} does not round to v0.3's {:.4f} - aborting.".format(got, exp)
print("bootstrap implementation check: PASS (same index stream, same patient grouping as v0.3)")
del _boot9, _d9


# =====================================================================================
# PART B - LOCKED QUANTITIES
# =====================================================================================
hdr("PART B - LOCKED QUANTITIES (refit on folds 1-9)")

# --- (a) categorical encoder refit on folds 1-9 ---
ENC19, CATS19 = {}, {}
for col in ACQ_COLS:
    ENC19[col], CATS19[col] = encode_categorical(LEVELS[col], mask_final, col)
WIDTHS18 = {col: ENC18[col].shape[1] for col in ACQ_COLS}
WIDTHS19 = {col: ENC19[col].shape[1] for col in ACQ_COLS}
NEW_LEVELS = {col: sorted(set(CATS19[col]) - set(CATS18[col])) for col in ACQ_COLS}
DROPPED_LEVELS = {col: sorted(set(CATS18[col]) - set(CATS19[col])) for col in ACQ_COLS}
assert all(len(v) == 0 for v in DROPPED_LEVELS.values()), "a folds 1-8 level vanished in folds 1-9 - aborting."
META_DIM_ALL_FINAL = len(B1_cols) + sum(WIDTHS19.values())
EXPECTED_META_DIM_FINAL = {"signal_only": 0, "signal_plus_physio": len(B1_cols), "signal_plus_all": META_DIM_ALL_FINAL}
HEAD_LINEAR_IN_FINAL = {arm: EMBEDDING_DIM + d for arm, d in EXPECTED_META_DIM_FINAL.items()}
META_DIM_ALL_DEV = len(B1_cols) + sum(WIDTHS18.values())
HEAD_LINEAR_IN_DEV = {"signal_only": EMBEDDING_DIM, "signal_plus_physio": EMBEDDING_DIM + 3,
                      "signal_plus_all": EMBEDDING_DIM + META_DIM_ALL_DEV}
assert HEAD_LINEAR_IN_DEV == {"signal_only": 256, "signal_plus_physio": 259, "signal_plus_all": 338}, \
    "development widths differ from dev_protocol.json 256/259/338 - aborting."
B3_COLS_FINAL = B1_cols + [c for col in ACQ_COLS for c in ENC19[col].columns]
assert len(B3_COLS_FINAL) == META_DIM_ALL_FINAL

print("{:8s} {:>12s} {:>12s}   new levels in folds 1-9".format("column", "folds 1-8", "folds 1-9"))
for col in ACQ_COLS:
    print("{:8s} {:>12d} {:>12d}   {}".format(col, WIDTHS18[col], WIDTHS19[col], NEW_LEVELS[col] or "-"))
print("meta dims  development (folds 1-8): signal_only 0 | signal_plus_physio 3 | signal_plus_all {}".format(META_DIM_ALL_DEV))
print("meta dims  FINAL       (folds 1-9): signal_only 0 | signal_plus_physio 3 | signal_plus_all {}".format(META_DIM_ALL_FINAL))
print("head first-Linear input widths FINAL:", HEAD_LINEAR_IN_FINAL)

# one-hot columns that are all-zero in training (zero gradient; only weight decay acts on them)
ALLZERO_TRAIN_COLS_19 = sorted(c for col in ACQ_COLS for c in ENC19[col].columns
                               if int(ENC19[col][c].sum()) == 0)
ALLZERO_TRAIN_COLS_18 = sorted(c for col in ACQ_COLS for c in ENC18[col].columns
                               if int(ENC18[col].loc[mask_train, c].sum()) == 0)
print("one-hot columns all-zero in training: folds 1-8 fit {} | folds 1-9 fit {}".format(
    ALLZERO_TRAIN_COLS_18, ALLZERO_TRAIN_COLS_19))
assert ALLZERO_TRAIN_COLS_19 == sorted(EXPECTED_ALLZERO_TRAIN_COLS_FOLDS1TO9), \
    "CROSS-CHECK FAILED: all-zero training columns (folds 1-9) {} != expected {} - STOP and report.".format(
        ALLZERO_TRAIN_COLS_19, sorted(EXPECTED_ALLZERO_TRAIN_COLS_FOLDS1TO9))

# fold-9 records hitting an all-zero column under the folds 1-8 encoder (affects v0.7 signal_plus_all)
FOLD9_UNTRAINED_DEV = {}
for col in ACQ_COLS:
    mapped9 = remap_with_categories(LEVELS[col][mask_eval], CATS18[col])
    names = "{}=".format(col) + mapped9
    FOLD9_UNTRAINED_DEV[col] = int(names.isin(ALLZERO_TRAIN_COLS_18).sum())
print("fold-9 records on an all-zero (untrained) column under the folds 1-8 encoder:", FOLD9_UNTRAINED_DEV)

# --- (b) age_imputed StandardScaler refit on folds 1-9 ---
age_scaler_18 = StandardScaler().fit(age_imputed[mask_train].to_numpy().reshape(-1, 1))
assert abs(float(age_scaler_18.mean_[0]) - V05_AGE_SCALER_FOLDS1TO8["mean"]) <= AGE_SCALER_TOL
assert abs(float(age_scaler_18.scale_[0]) - V05_AGE_SCALER_FOLDS1TO8["scale"]) <= AGE_SCALER_TOL
age_scaler_19 = StandardScaler().fit(age_imputed[mask_final].to_numpy().reshape(-1, 1))
AGE_SCALER_FINAL = {"mean": float(age_scaler_19.mean_[0]), "scale": float(age_scaler_19.scale_[0]),
                    "var": float(age_scaler_19.var_[0]), "n_samples_seen": int(age_scaler_19.n_samples_seen_)}
print("age_imputed scaler: folds 1-8 mean={:.4f} scale={:.4f} (== v0.5: PASS) | FINAL folds 1-9 mean={!r} scale={!r}".format(
    float(age_scaler_18.mean_[0]), float(age_scaler_18.scale_[0]), AGE_SCALER_FINAL["mean"], AGE_SCALER_FINAL["scale"]))

# --- (c) per-lead normalization refit on folds 1-9 (two-pass float64, v0.4 code) ---
_old_idx = pd.read_csv(SIGNALS_INDEX, encoding="utf-8-sig")
assert list(_old_idx.columns) == list(current_index.columns) and _old_idx.astype(str).reset_index(drop=True).equals(
    current_index.astype(str).reset_index(drop=True)), "v0.4 sidecar index does not match folds 1-9 - aborting."
t0 = time.time()
X = np.load(SIGNALS_NPY)
assert X.shape == (N_REC, N_LEADS, N_SAMPLES) and X.dtype == np.float32


def lead_norm_two_pass(X, rows, chunk=CHUNK):
    """Per-lead mean / population std over the given rows; float64 two-pass in chunks of
    `chunk` records in row order (v0.4 Part B code, unchanged)."""
    _sum = np.zeros(N_LEADS, dtype=np.float64)
    for s in range(0, len(rows), chunk):
        _sum += X[rows[s:s + chunk]].astype(np.float64).sum(axis=(0, 2))
    _n_vals = len(rows) * N_SAMPLES
    mean = _sum / _n_vals
    _sq = np.zeros(N_LEADS, dtype=np.float64)
    for s in range(0, len(rows), chunk):
        xc = X[rows[s:s + chunk]].astype(np.float64) - mean[None, :, None]
        _sq += (xc ** 2).sum(axis=(0, 2))
    std = np.sqrt(_sq / _n_vals)
    return mean, std


LEAD_MEAN_18, LEAD_STD_18 = lead_norm_two_pass(X, train_rows)
_v04_norm = pd.read_csv(V04_LEAD_NORM_CSV, encoding="utf-8-sig")
assert np.allclose(_v04_norm["mean"].to_numpy(), LEAD_MEAN_18, rtol=0, atol=1e-12) and \
    np.allclose(_v04_norm["std"].to_numpy(), LEAD_STD_18, rtol=0, atol=1e-12), \
    "folds 1-8 lead normalization differs from v0.4_lead_norm_params.csv - aborting."
LEAD_NAMES = _v04_norm["lead"].tolist()
LEAD_MEAN_19, LEAD_STD_19 = lead_norm_two_pass(X, all_rows)
assert np.all(LEAD_STD_19 > 0) and np.all(np.isfinite(LEAD_MEAN_19))
del X
gc.collect()
print("per-lead normalization: folds 1-8 recomputed == v0.4_lead_norm_params.csv (atol 1e-12): PASS | "
      "folds 1-9 computed over {} records ({:.1f}s)".format(N_REC, time.time() - t0))
print(pd.DataFrame({"lead": LEAD_NAMES, "mean_folds1to9": LEAD_MEAN_19, "std_folds1to9": LEAD_STD_19,
                    "mean_folds1to8": LEAD_MEAN_18, "std_folds1to8": LEAD_STD_18}).to_string(index=False))

# --- OneCycle step counts and the weight-decay shrink factor of untrained columns ---
N_BATCHES_FINAL = math.ceil(N_REC / EXPECTED_BATCH_SIZE["resnet1d_wang"])
assert BATCH_SIZE["resnet1d_wang"] == BATCH_SIZE["inception1d"] == 128
assert N_BATCHES_FINAL == EXPECTED_N_BATCHES_FINAL and EPOCHS * N_BATCHES_FINAL == EXPECTED_TOTAL_STEPS_FINAL
assert N_REC % 128 == EXPECTED_LAST_BATCH_FINAL, "last batch size {} != 17".format(N_REC % 128)
N_BATCHES_DEV = math.ceil(len(train_rows) / 128)
assert N_BATCHES_DEV == EXPECTED_N_BATCHES_DEV and EPOCHS * N_BATCHES_DEV == EXPECTED_TOTAL_STEPS_DEV
TOTAL_STEPS_FINAL = EPOCHS * N_BATCHES_FINAL
TOTAL_STEPS_DEV = EPOCHS * N_BATCHES_DEV


def onecycle_lrs(max_lr, total_steps, pct_start, div_factor, final_div_factor):
    """lr used by the optimizer at steps 0..total_steps-1, replicating
    torch.optim.lr_scheduler.OneCycleLR(three_phase=False, anneal_strategy='cos')."""
    initial_lr = max_lr / div_factor
    min_lr = initial_lr / final_div_factor
    phases = [(float(pct_start * total_steps) - 1, initial_lr, max_lr), (total_steps - 1, max_lr, min_lr)]
    out = np.empty(total_steps, dtype=np.float64)
    for step_num in range(total_steps):
        start_step = 0.0
        for i, (end_step, start_lr, end_lr) in enumerate(phases):
            if step_num <= end_step or i == len(phases) - 1:
                pct = (step_num - start_step) / (end_step - start_step)
                out[step_num] = end_lr + (start_lr - end_lr) / 2.0 * (math.cos(math.pi * pct) + 1)
                break
            start_step = end_step
    return out


SHRINK = {}
for label, steps in (("development_folds1to8", TOTAL_STEPS_DEV), ("final_folds1to9", TOTAL_STEPS_FINAL)):
    lrs = onecycle_lrs(SELECTED_LR["resnet1d_wang"], steps, ONECYCLE["pct_start"], ONECYCLE["div_factor"],
                       ONECYCLE["final_div_factor"])
    assert abs(lrs.max() - SELECTED_LR["resnet1d_wang"]) < 1e-12
    SHRINK[label] = {"total_steps": steps, "sum_lr": float(lrs.sum()),
                     "shrink_factor_exact": float(np.prod(1.0 - WEIGHT_DECAY * lrs)),
                     "shrink_factor_exp_approx": float(math.exp(-WEIGHT_DECAY * lrs.sum()))}
    print("untrained-column weight shrink ({:22s}): steps {} | sum(lr) {:.4f} | prod(1 - wd*lr) = {:.4f} "
          "(exp approx {:.4f})".format(label, steps, SHRINK[label]["sum_lr"], SHRINK[label]["shrink_factor_exact"],
                                       SHRINK[label]["shrink_factor_exp_approx"]))

# --- (d) fold-10 METADATA counts (allowed columns only; counts only, no record identities) ---
fold10_pid = fold10_meta["patient_id"].to_numpy()
assert np.all(fold10_pid == np.round(fold10_pid)), "non-integral fold-10 patient_id"
fold10_pid = fold10_pid.astype(np.int64)
fold10_ecg = fold10_meta.index.to_numpy().astype(np.int64)
N_F10 = len(fold10_meta)
N_F10_PATIENTS = int(len(np.unique(fold10_pid)))
F10_DISJOINT = set(fold10_pid.tolist()).isdisjoint(set(df9["patient_id"].astype(np.int64).tolist()))
assert N_F10 == EXPECTED_N_FOLD10_RECORDS, "fold-10 records {} != 2198 - aborting.".format(N_F10)
assert N_F10_PATIENTS == EXPECTED_N_FOLD10_PATIENTS, "fold-10 patients {} != 1904 - aborting.".format(N_F10_PATIENTS)
assert F10_DISJOINT, "a fold-10 patient appears in folds 1-9 - aborting."

F10_COUNTS = {"n_unknown": {}, "n_on_untrained_column": {}}
_any_unknown = np.zeros(N_F10, dtype=bool)
_any_untrained = np.zeros(N_F10, dtype=bool)
for col in ACQ_COLS:
    joint = to_level_string(pd.concat([df9[col], fold10_meta[col]]))
    assert (joint.iloc[:N_REC].to_numpy() == LEVELS[col].to_numpy()).all(), \
        "{}: level strings of folds 1-9 change when fold-10 values are present (zero-pad width) - aborting.".format(col)
    lvl10 = joint.iloc[N_REC:]
    mapped10 = remap_with_categories(lvl10, CATS19[col])
    is_unknown = (mapped10 == "__UNKNOWN__").to_numpy()
    is_untrained = ("{}=".format(col) + mapped10).isin(ALLZERO_TRAIN_COLS_19).to_numpy()
    F10_COUNTS["n_unknown"][col] = int(is_unknown.sum())
    F10_COUNTS["n_on_untrained_column"][col] = int(is_untrained.sum())
    _any_unknown |= is_unknown
    _any_untrained |= is_untrained
F10_COUNTS["n_unknown"]["any"] = int(_any_unknown.sum())
F10_COUNTS["n_on_untrained_column"]["any"] = int(_any_untrained.sum())
del _any_unknown, _any_untrained, lvl10, mapped10, is_unknown, is_untrained, joint
print("fold 10: n records {} | n patients {} | disjoint from folds 1-9 patients: {}".format(
    N_F10, N_F10_PATIENTS, "PASS" if F10_DISJOINT else "FAIL"))
print("fold 10 under the folds 1-9 encoder: __UNKNOWN__ counts {} | records on any all-zero training column {}".format(
    F10_COUNTS["n_unknown"], F10_COUNTS["n_on_untrained_column"]))
assert F10_COUNTS["n_on_untrained_column"]["any"] == EXPECTED_FOLD10_UNTRAINED_COUNT and \
    F10_COUNTS["n_unknown"]["any"] == 0, \
    "CROSS-CHECK FAILED: fold-10 records map to __UNKNOWN__ / untrained columns - STOP and report."

# --- fold-10 bootstrap and permutation index matrices (patient_id / record count only) ---
t0 = time.time()
F10_PATIENTS, F10_ROWS = REF["patient_rows_in_file_order"](fold10_pid)
assert len(F10_PATIENTS) == N_F10_PATIENTS and sum(len(r) for r in F10_ROWS) == N_F10
BOOT_IDX = REF["build_boot_idx"](N_F10_PATIENTS, N_BOOT, BOOT_SEED)
PERM_IDX = REF["build_perm_idx"](N_F10, N_PERM, PERM_SEED)
assert BOOT_IDX.shape == (N_BOOT, N_F10_PATIENTS) and PERM_IDX.shape == (N_PERM, N_F10)
F10_IDS = np.ascontiguousarray(np.stack([fold10_ecg, fold10_pid], axis=1).astype("<i8"))
LOCK_HASHES = {
    "fold10_ids_file_order_sha256": sha256_bytes(F10_IDS.tobytes(order="C")),
    "fold10_patients_sorted_sha256": sha256_bytes(np.ascontiguousarray(F10_PATIENTS.astype("<i8")).tobytes()),
    "boot_idx_sha256": sha256_bytes(np.ascontiguousarray(BOOT_IDX.astype("<i8")).tobytes()),
    "perm_idx_sha256": sha256_bytes(np.ascontiguousarray(PERM_IDX.astype("<i8")).tobytes()),
}
# re-generation is deterministic (what v0.9 will do)
assert sha256_bytes(np.ascontiguousarray(REF["build_boot_idx"](N_F10_PATIENTS, N_BOOT, BOOT_SEED).astype("<i8")).tobytes()) \
    == LOCK_HASHES["boot_idx_sha256"]
assert sha256_bytes(np.ascontiguousarray(REF["build_perm_idx"](N_F10, N_PERM, PERM_SEED).astype("<i8")).tobytes()) \
    == LOCK_HASHES["perm_idx_sha256"]
_draw_sizes = np.array([sum(len(F10_ROWS[i]) for i in BOOT_IDX[b]) for b in range(N_BOOT)])
print("fold-10 index matrices built in {:.1f}s: BOOT_IDX {} | PERM_IDX {} | records per bootstrap draw: "
      "min {} / median {:.0f} / max {}".format(time.time() - t0, BOOT_IDX.shape, PERM_IDX.shape,
                                               _draw_sizes.min(), np.median(_draw_sizes), _draw_sizes.max()))
for k, v in LOCK_HASHES.items():
    print("  {:32s} {}".format(k, v))
del BOOT_IDX, PERM_IDX, F10_IDS, _draw_sizes

# --- H3 verdict function: truth-table self-test (synthetic CI bounds, not data) ---
_cases = [  # (lo95, hi95, lo90, hi90) -> expected verdict, a, b, material
    ((0.002, 0.008, 0.003, 0.007), "statistically detectable, below the materiality threshold", True, True, False),
    ((0.012, 0.030, 0.014, 0.028), "acquisition metadata materially changes performance (sign: +)", True, False, True),
    ((-0.030, -0.012, -0.028, -0.014), "acquisition metadata materially changes performance (sign: -)", True, False, True),
    ((0.002, 0.020, 0.004, 0.018), "acquisition metadata changes performance (sign: +); magnitude relative to SESOI undetermined", True, False, False),
    ((-0.005, 0.005, -0.004, 0.004), "no material change", False, True, False),
    ((-0.012, 0.009, -0.010, 0.010), "no material change", False, True, False),      # closed interval
    ((-0.005, 0.020, -0.003, 0.018), "inconclusive", False, False, False),
    ((0.0, 0.010, 0.001, 0.009), "no material change", False, True, False),           # lo95 == 0 -> a fails
    ((0.010, 0.030, 0.012, 0.028), "acquisition metadata changes performance (sign: +); magnitude relative to SESOI undetermined", True, False, False),
]
for (lo95, hi95, lo90, hi90), v, a, b, m in _cases:
    r = REF["h3_verdict"](lo95, hi95, lo90, hi90, SESOI_H3)
    assert (r["verdict"], r["a_holds"], r["b_holds"], r["material"]) == (v, a, b, m), (lo95, hi95, lo90, hi90, r)
print("h3_verdict truth-table self-test ({} cases): PASS".format(len(_cases)))


# =====================================================================================
# PART C - final_run_protocol.json
# =====================================================================================
hdr("PART C - final_run_protocol.json")

LOCKED_QUANTITIES = {
    "fit_rows": "folds 1-9 of PTB-XL v1.0.3 (strat_fold <= 9): {} records".format(N_REC),
    "categorical_encoder": {
        "procedure": ("to_level_string (v0.2.1, zero-padded numeric levels) then encode_categorical with "
                      "train_mask = all folds 1-9 rows; categories = sorted(training levels | {'__MISSING__'}) + "
                      "['__UNKNOWN__']; any level not among the training categories -> '__UNKNOWN__'"),
        "categories": {col: CATS19[col] for col in ACQ_COLS},
        "widths": WIDTHS19,
        "widths_folds1to8_development": WIDTHS18,
        "new_levels_vs_folds1to8": NEW_LEVELS,
        "signal_plus_all_column_order": B3_COLS_FINAL,
        "allzero_training_columns_folds1to9": ALLZERO_TRAIN_COLS_19,
        "allzero_training_columns_folds1to8": ALLZERO_TRAIN_COLS_18,
        "fold10_level_mapping": ("for each of device/site/nurse: to_level_string applied to the concatenation "
                                 "[folds 1-9 column, fold-10 column]; the folds 1-9 part must equal the folds 1-9-only "
                                 "level strings (asserted in v0.6); the fold-10 part is remapped with the categories "
                                 "above (same rule as encode_categorical)"),
    },
    "physio_columns": {"order": B1_cols, "age_sentinel": "age == 300 -> age_censored = 1, age_imputed = 90; else "
                       "age_censored = 0, age_imputed = age", "sex": "raw 0/1", "age_imputed": "standardized with age_scaler"},
    "meta_dim": EXPECTED_META_DIM_FINAL,
    "head_first_linear_in_features": HEAD_LINEAR_IN_FINAL,
    "meta_dim_development_folds1to8": {"signal_only": 0, "signal_plus_physio": 3, "signal_plus_all": META_DIM_ALL_DEV},
    "age_scaler": {"fit_on": "age_imputed, folds 1-9 (sklearn StandardScaler, population std)", **AGE_SCALER_FINAL},
    "lead_normalization": {
        "fit_on": "v0.4 signal cache, all {} folds 1-9 records, sha256 {}".format(N_REC, EXPECTED_SIGNALS_NPY_SHA256),
        "procedure": ("two-pass float64: pass 1 sums per lead over chunks of 1000 records in cache row order, "
                      "mean = sum / (n_records * 1000); pass 2 sums squared deviations the same way; "
                      "std = sqrt(ss / (n_records * 1000)) (population std); applied as (x - mean) / std in float32"),
        "lead_order": LEAD_NAMES,
        "mean": [float(v) for v in LEAD_MEAN_19],
        "std": [float(v) for v in LEAD_STD_19],
        "comparison_rule_for_v0.8_v0.9": "recomputed values must equal these with atol 1e-12 (as v0.5 did)",
    },
    "final_training_steps": {"n_records": N_REC, "batch_size": 128, "n_batches_per_epoch": N_BATCHES_FINAL,
                             "last_batch_size": N_REC % 128, "epochs": EPOCHS, "onecycle_total_steps": TOTAL_STEPS_FINAL},
    "development_training_steps": {"n_records": len(train_rows), "n_batches_per_epoch": N_BATCHES_DEV,
                                   "onecycle_total_steps": TOTAL_STEPS_DEV},
    "untrained_column_weight_shrink": SHRINK,
    "fold9_records_on_untrained_columns_under_folds1to8_encoder": FOLD9_UNTRAINED_DEV,
    "fold10": {
        "n_records": N_F10, "n_patients": N_F10_PATIENTS, "patient_disjoint_from_folds1to9": bool(F10_DISJOINT),
        "n_unknown_under_folds1to9_encoder": F10_COUNTS["n_unknown"],
        "n_on_untrained_column_under_folds1to9_encoder": F10_COUNTS["n_on_untrained_column"],
        "columns_read_in_v0.6": FOLD10_COLUMNS_KEPT,
    },
}

ENCODER_WIDTH_NOTE = (
    "The 256/259/338 widths in dev_protocol.json 'metadata_fusion' describe the folds 1-8 fit used in development "
    "(device/site/nurse one-hot widths {}/{}/{}, signal_plus_all meta_dim {}). The sealed principle 'encoder fit on "
    "training folds only' is applied in the final run by refitting on folds 1-9, which gives one-hot widths "
    "{}/{}/{} (new levels: {}), meta_dim {} / {} / {} and head first-Linear input widths {} / {} / {} for "
    "signal_only / signal_plus_physio / signal_plus_all. Fusion location, raw metadata entry, and everything else "
    "in 'metadata_fusion' are unchanged.").format(
        WIDTHS18["device"], WIDTHS18["site"], WIDTHS18["nurse"], META_DIM_ALL_DEV,
        WIDTHS19["device"], WIDTHS19["site"], WIDTHS19["nurse"],
        "; ".join("{}: {}".format(c, NEW_LEVELS[c]) for c in ACQ_COLS if NEW_LEVELS[c]) or "none",
        0, 3, META_DIM_ALL_FINAL,
        HEAD_LINEAR_IN_FINAL["signal_only"], HEAD_LINEAR_IN_FINAL["signal_plus_physio"],
        HEAD_LINEAR_IN_FINAL["signal_plus_all"])

UNTRAINED_COLUMNS_NOTE = (
    "One-hot columns that are all-zero in training (__UNKNOWN__, and __MISSING__ where absent in training: "
    "folds 1-8 fit {}; folds 1-9 fit {}) receive zero gradient, but AdamW's decoupled weight decay only shrinks "
    "their first-Linear weights multiplicatively: prod over steps of (1 - wd * lr_step) = {:.4f} over {} "
    "development steps and {:.4f} over {} final steps (computed from the actual OneCycle lr schedule with "
    "max_lr 0.01). So ~{:.0f}% of the random initial weight remains: a record mapped to such a column gets a small "
    "random input shift in signal_plus_all. Affects fold-9 signal_plus_all results in v0.7 ({} fold-9 records under "
    "the folds 1-8 encoder: {}). Under the folds 1-9 encoder, {} fold-10 records map to such a column "
    "(cross-check: expected 0).").format(
        ALLZERO_TRAIN_COLS_18, ALLZERO_TRAIN_COLS_19,
        SHRINK["development_folds1to8"]["shrink_factor_exact"], TOTAL_STEPS_DEV,
        SHRINK["final_folds1to9"]["shrink_factor_exact"], TOTAL_STEPS_FINAL,
        100 * SHRINK["final_folds1to9"]["shrink_factor_exact"],
        sum(FOLD9_UNTRAINED_DEV.values()), FOLD9_UNTRAINED_DEV, F10_COUNTS["n_on_untrained_column"]["any"])

STAGE_PLAN = {
    "v0.7": (
        "EXPLORATORY development matrix on fold 9: 2 architectures (resnet1d_wang, inception1d) x 3 arms "
        "(signal_only, signal_plus_physio, signal_plus_all) x 3 seeds (42, 123, 2024) at the locked lr 0.01; training "
        "on folds 1-8, fold-9 evaluation once at the end of epoch 50; recipe, folds 1-8 encoder (widths 256/259/338), "
        "age scaler and lead normalization exactly as dev_protocol.json, v0.5_training_details.json and the v0.5 code. "
        "Reused, NOT retrained (sha256-verified against v0.5_manifest.json, file sha256 {}; per-file hashes in "
        "v0.7_reused_v0.5_files): v0.5 resnet1d_wang signal_plus_physio seed 42 lr 0.01, v0.5 inception1d "
        "signal_plus_physio seed 42 lr 0.01, v0.5 resnet1d_wang signal_only seed 42 lr 0.01 -> 15 new runs. "
        "Status: EXPLORATORY; no setting may be chosen or changed from it. Known bias: lr was selected on fold 9 with "
        "signal_plus_physio, so on fold 9 the physio - signal_only delta is biased UPWARD and the all - physio delta "
        "DOWNWARD.").format(V05_MAN_SHA),
    "v0.8": (
        "final training on folds 1-9: 2 architectures x 3 arms x 3 seeds = 18 runs, lr 0.01, 50 epochs, everything "
        "else exactly as v0.5_training_details.json with the v0.8_overrides below; encoder, age scaler and lead "
        "normalization = the locked folds 1-9 values in locked_quantities (asserted equal at v0.8 STEP 0). "
        "signal_only is trained as an EXPLORATORY arm (used only in exploratory_fold10_comparisons). NO evaluation "
        "of any kind. v0.8 opens no fold-10 signal file and builds no fold-10 tensor (asserted). Saves per run the "
        "final state_dict, the training-loss CSV, and the sha256 of both in the v0.8 manifest."),
    "v0.9": (
        "the single opening of fold 10 (effort = Max). First and only stage that reads fold-10 signals or labels; "
        "asserted by code. STEP 0 verifies this document, regenerates BOOT_IDX / PERM_IDX / the fold-10 id list with "
        "reference_implementation and asserts the locked hashes. Evaluation runs FROM THE SAVED, HASH-VERIFIED v0.8 "
        "state_dicts (CUDA kernels are not bitwise deterministic, so no model is re-trained or re-created by "
        "training). Evaluation = 7 windows, mean of sigmoid probabilities per record, exactly as v0.5, with the "
        "locked folds 1-9 lead normalization, age scaler and encoder applied to fold 10. The fold-10 probabilities "
        "of all 18 models are written to files and their sha256 recorded BEFORE any metric is computed. Fold-10 "
        "labels are derived with the v0.2.1 label code (verbatim)."),
}

V08_OVERRIDES = {
    "train_data": "folds 1-9 (19601 records), RandomCropDataset as v0.5; per-lead normalization = locked folds 1-9 values",
    "metadata_encoding": "categorical encoder and age_imputed scaler = locked folds 1-9 values",
    "not_applicable_in_v0.8": ["validation", "evaluation", "metric", "lr_tie_break", "per_run_outputs: fold9_probs file"],
    "per_run_outputs_v0.8": "final state_dict and train_loss csv, sha256 of both in the v0.8 manifest",
    "lr_scheduler_total_steps_v0.8": TOTAL_STEPS_FINAL,
    "applies_unchanged": ["optimizer", "lr_scheduler (except total_steps)", "epochs", "batch_size", "data_order",
                          "seeding", "training_loss", "training_loss_logging", "determinism_note",
                          "explicit rule (from 'validation'): no checkpoint selection; the model after the last "
                          "optimizer step is final"],
    "v0.9_evaluation": ("the 'evaluation' and 'metric' fields apply to fold-10 records in v0.9, with 'folds 1-8 "
                        "statistics' replaced by the locked folds 1-9 lead normalization and age scaler"),
}

CRASH_AND_RERUN_RULES = {
    "v0.7": ("a run may be re-run from scratch with the same seed only if it crashed or produced a non-finite "
             "training loss BEFORE its fold-9 evaluation; a run whose fold-9 evaluation was computed is never re-run; "
             "every attempt (including failed ones) is logged in the v0.7 manifest"),
    "v0.8": ("because v0.8 performs no evaluation, a run that crashes or yields a non-finite loss may be re-run from "
             "scratch with the same seed BEFORE v0.9; every attempt is logged in the v0.8 manifest. If the re-run fails "
             "again: stop and report; no setting may be changed except through a written, externally timestamped "
             "amendment made before v0.9. Once v0.9 has started, no re-run is allowed."),
    "v0.9": "re-execution only as in post_fold10_rules 2",
}

POST_FOLD10_RULES = [
    "1. No model that enters the H3 analysis or any comparison listed in this file is trained after fold 10 is opened.",
    "2. Any re-execution of v0.9 uses only the saved, sha256-verified v0.8 state_dicts and the saved, sha256-verified "
    "fold-10 probability files.",
    "3. Ablations listed in the project plan that require training (random-split ablation; height/weight sensitivity) "
    "are POST-HOC studies: they run under their own version ids, never modify or overwrite any H3 / listed-comparison "
    "artifact, and are reported separately from pre-registered results.",
    "4. The device/site permutation ablation (device + site + nurse one-hot block; specification in "
    "exploratory_fold10_comparisons) is done at inference time on the saved v0.8 state_dicts (permutation of metadata "
    "input), exploratory.",
]

BOOTSTRAP = {
    "data": "fold 10 ({} records), labels NORM/MI/STTC/CD/HYP derived with the v0.2.1 code".format(N_F10),
    "patients": ("patients = np.unique(fold-10 patient_id as int64) (sorted); patient_row_list[i] = array of fold-10 "
                 "row positions of patients[i], rows in fold-10 file order (ptbxl_database.csv row order); "
                 "reference_implementation.patient_rows_in_file_order"),
    "index_matrix": ("BOOT_IDX is built by N_BOOT sequential calls rng.integers(0, n_patients, size=n_patients) on one "
                     "Generator(BOOT_SEED=7), row b = call b, exactly as v0.3; a single call with size=(N_BOOT, "
                     "n_patients) is not used (reference_implementation.build_boot_idx). Each draw: rows = "
                     "np.concatenate([patient_row_list[i] for i in BOOT_IDX[b]])."),
    "N_BOOT": N_BOOT, "BOOT_SEED": BOOT_SEED, "n_patients": N_F10_PATIENTS,
    "sharing": "BOOT_IDX is generated ONCE and shared by every comparison, every seed and both architectures (fully paired)",
    "per_draw": ("macro-AUROC of each arm = mean over NORM, MI, STTC, CD, HYP of fast_auc (v0.2.1 function) on the "
                 "drawn rows; delta = arm_B - arm_A (H3: arm_B = signal_plus_all, arm_A = signal_plus_physio)"),
    "nan_rule": ("if a class has no positives or no negatives in a draw, that draw's macro delta is NaN; it is excluded "
                 "(not redrawn) and the count is reported; if > 1% of draws are NaN this is flagged in the report"),
    "point_estimate": "delta on the full fold 10 (not the bootstrap mean)",
    "ci": ("percentile, np.percentile(valid_deltas, q, method='linear'); 95% = [2.5, 97.5], 90% = [5, 95]; "
           "no rounding before any comparison"),
    "primary_input": ("per-record probabilities averaged (arithmetic mean) over seeds 42, 123, 2024 per arm and per "
                      "architecture, then macro-AUROC. Per-seed deltas (same BOOT_IDX) are secondary."),
    "locked_hashes": {
        **LOCK_HASHES,
        "serialization": ("fold10_ids: (n, 2) array [ecg_id, patient_id] in fold-10 file order, cast to '<i8', "
                          "C-order bytes; fold10_patients: np.unique(patient_id) as '<i8' bytes; boot_idx: "
                          "BOOT_IDX ({} x {}) as '<i8' C-order bytes; perm_idx: PERM_IDX ({} x {}) as "
                          "'<i8' C-order bytes".format(N_BOOT, N_F10_PATIENTS, N_PERM, N_F10)),
        "v0.9_rule": "v0.9 regenerates all four with reference_implementation and asserts equality before any metric",
    },
    "deviation_from_v0.3": ("v0.3 excluded a NaN class from the macro mean (per-class nan-skip); this protocol excludes "
                            "the whole draw. In v0.3 (fold 9) no draw was affected (0 invalid replicates)."),
    "implementation_check_in_v0.6": BOOT_IMPL_CHECK,
}

H3_DECISION = {
    "status": "resolution of the overlap between sealed rules a and b; written before fold 10 is opened",
    "definitions": {
        "a_holds": "95% CI excludes 0 (lo95 > 0 or hi95 < 0)",
        "b_holds": "90% CI entirely within the CLOSED interval [-0.01, +0.01] (lo90 >= -0.01 and hi90 <= 0.01)",
        "material": "95% CI entirely outside [-0.01, +0.01] (lo95 > 0.01 or hi95 < -0.01)",
        "sign": "'+' if lo95 > 0, '-' if hi95 < 0 (the side of 0 the 95% CI lies on)",
    },
    "verdict_table": {
        "a and b": "statistically detectable, below the materiality threshold",
        "a and not b, material": "acquisition metadata materially changes performance (sign: +/-)",
        "a and not b, not material": "acquisition metadata changes performance (sign: +/-); magnitude relative to SESOI undetermined",
        "not a and b": "no material change",
        "not a and not b": "inconclusive",
    },
    "SESOI_H3": SESOI_H3,
    "rationale": {
        "materiality": ("materiality uses the 95% CI because H3 is two-sided and a materiality claim is a union over "
                        "two directions (alpha split 0.025 per side)"),
        "equivalence": "the equivalence condition (b) uses the 90% CI because TOST is an intersection-union test",
    },
    "reporting_rule": ("the sealed labels a/b/c are always reported verbatim (sealed_H3_protocol_verbatim.decision) next "
                       "to the verdict, together with a_holds, b_holds, material, the point estimate and both CIs"),
    "implementation": "reference_implementation.h3_verdict (truth table self-tested in v0.6)",
}

REPORTING_ORDER = [
    "1. Primary verdict: resnet1d_wang, seed-averaged probabilities.",
    "2. Robustness: inception1d, same procedure, same verdict table.",
    "3. Secondary: per-seed deltas for both architectures.",
    "4. Disagreements between architectures or between seed-averaged and per-seed results are reported as they are; "
    "none is resolved by selection.",
    "5. Exploratory results after all of the above.",
]

EXPLORATORY_COMPARISONS = {
    "rule": "closed list; anything else is post-hoc and labelled so",
    "physio_minus_signal_only": ("signal_plus_physio - signal_only: both architectures, seed-averaged probabilities, "
                                 "same BOOT_IDX, point estimate, 95% and 90% percentile CIs; no decision label. Purpose: "
                                 "unbiased estimate of the age/sex effect. (No signal_plus_all - signal_only comparison.)"),
    "per_class_deltas": ("per-class deltas (NORM, MI, STTC, CD, HYP) for the H3 pair (signal_plus_all - "
                         "signal_plus_physio), both architectures, seed-averaged, same BOOT_IDX, point estimate and 95% "
                         "percentile CI per class (a per-class draw is NaN only if that class is degenerate in the draw); "
                         "no multiplicity correction, no decision label"),
    "acquisition_permutation": {
        "what": ("signal_plus_all, both architectures; joint permutation of the device + site + nurse one-hot block "
                 "across fold-10 records (rows of the block are permuted together; age_imputed, age_censored and sex "
                 "stay fixed); permuted metadata row i = acquisition block of record PERM_IDX[r, i]"),
        "repetitions": ("N_PERM = {} sequential rng.permutation({}) calls on one default_rng(PERM_SEED = {}) "
                        "(reference_implementation.build_perm_idx); PERM_IDX hash locked in bootstrap.locked_hashes"
                        ).format(N_PERM, N_F10, PERM_SEED),
        "seeds": ("in each repetition the SAME permutation is applied to all three seed models; their probabilities "
                  "are averaged, then macro-AUROC"),
        "report": "mean and range (min, max) over the {} repetitions of drop = macro-AUROC(unpermuted) - macro-AUROC(permuted); no CI, no decision label".format(N_PERM),
        "timing": "inference only, on the saved v0.8 state_dicts (post_fold10_rules 4)",
    },
}

DISCLOSURES = [
    ("Fold-10 metadata (device/site/nurse distributions) was examined in the audit stages (details in "
     "disclosures_fold10_exposure); v0.6 kept only ecg_id, patient_id, device, site and nurse of the fold-10 rows in a "
     "variable (pd.read_csv parsed every column once; the other fold-10 columns were dropped immediately and never used)."),
    ("Selected lr 0.01 is the upper edge of the grid for both architectures; lr values differed by at most 0.0023 "
     "(resnet) / 0.0045 (inception) on one seed, i.e. within plausible seed noise."),
    ("The models are re-implementations from memory of helme/ecg_ptbxl_benchmarking; they will be compared to the "
     "repository before writing, and deviations reported as limitations; the architecture is NOT changed."),
    ("Per-record aggregation is the mean over windows (reference may use max), fixed a priori in dev_protocol.json."),
    "torch.use_deterministic_algorithms is not enabled.",
]

DISCLOSURES_FOLD10_EXPOSURE = {
    "summary": ("No metadata-label relationship and no model performance was ever computed on fold 10. Every "
                "whole-dataset or per-fold statistic in v0.1.1/v0.1.2 includes fold-10 rows. v0.1 (whose code would "
                "have computed superclass prevalence by device/site on the pooled dataset incl. fold 10) was revised "
                "into v0.1.1 before it was ever executed (confirmed by the user). v0.2 was never executed on real data; "
                "the death experiment ran as v0.2.1 with fold 10 asserted absent."),
    "fold10_labels": [
        ("marginal superclass counts/prevalence on fold 10 (NORM 963, MI 550, STTC 521, CD 496, HYP 262 of 2198), "
         "computed and printed in v0.1.1 and v0.1.2 and stored in both manifests"),
        ("v0.1.1_labels.csv and v0.1.2_labels.csv contain per-record superclass labels for all 21799 records incl. "
         "fold 10 (with patient_id, strat_fold); never analysed for fold 10. File sha256: v0.1.1 = {} (from its "
         "manifest; asserted in v0.6); v0.1.2 = {} (computed in v0.6 by sha256_of() on the file bytes only; the file "
         "was NOT parsed)").format(V011_LABELS_SHA, V012_LABELS_SHA),
        ("whole-dataset label statistics: superclass counts (NORM 9514, MI 5469, STTC 5235, CD 4898, HYP 2649), 411 "
         "records without superclass, labels-per-record distribution (5144 multi-label), scp_codes likelihood "
         "distribution, likelihood>0 sensitivity (127 records; STTC 5235 -> 5108)"),
        ("PTB-XL v1.0.2/v1.0.3 changelogs printed in full (public dataset documentation), which list SCP-code "
         "changes/consensus labels for a few fold-10 ecg_ids (e.g. 7783)"),
    ],
    "fold10_metadata_only": [
        "device / site / nurse x strat_fold crosstabs incl. fold 10, with Cramer's V (v0.1.1, v0.1.2)",
        "recording_year x strat_fold incl. fold 10, with Cramer's V (v0.1.2)",
        "validated_by_human x strat_fold (folds 9 and 10: 100% True)",
        ("fold sizes and patient counts (fold 10: 2198 records, 1904 patients); patient-disjointness check over all "
         "folds"),
        "unseen-category check: site_049 has 1 fold-10 record (absent in folds 1-8; 2 records in fold 9)",
    ],
    "whole_dataset_metadata": [
        ("distributions of device, site, nurse, recording year, sex, age (sentinel 300: 293 records overall vs 259 in "
         "folds 1-9), height/weight (missing 68.0% / 56.8%), column missingness, records per patient, metadata-patient "
         "entanglement, effective cardinality, device x recording_year"),
    ],
    "not_computed_on_fold10": ("superclass prevalence by device/site and the metadata-label permutation test (v0.1.1 "
                               "s7.4, v0.1.2 s7.4/7.7) are restricted to folds 1-9; no age/sex statistic by fold; no "
                               "model output"),
    "rerun_note": ("v0.1.2 was re-executed on 2026-10-02 (03:35 local) to supply outputs for this review; identical "
                   "computations, no new fold-10 quantity; its manifest sha256 is now {}").format(V012_MAN_SHA),
    "v0.1.x_file_hashes": {"v0.1.1_labels.csv": V011_LABELS_SHA, "v0.1.1_manifest.json": V011_MAN_SHA,
                           "v0.1.2_labels.csv": V012_LABELS_SHA, "v0.1.2_manifest.json": V012_MAN_SHA,
                           "v0.1.2_manifest_records_labels_sha256": bool(V012_MAN_RECORDS_LABELS)},
}

DOCUMENTS_TO_TIMESTAMP = [SEAL_PATH, DEV_PROTOCOL_PATH, TRAINING_DETAILS_PATH, LR_SELECTION_PATH, FINAL_PROTOCOL_PATH]

PROTOCOL = {
    "document": "final run and fold-10 analysis protocol for H3 (write-once)",
    "timestamp": datetime.now().astimezone().isoformat(),
    "code_version": CODE_VERSION,
    "parent_hashes": {key: {"self_sha256": DOC_SELF[key], "file_sha256": DOC_FILE[key]} for key in DOC_PATHS},
    "input_files_sha256": INPUT_HASHES,
    "v0.4_signal_cache_sha256": CACHE_HASHES,
    "earlier_manifests_sha256": {"v0.3_manifest.json": V03_MAN_SHA, "v0.4_manifest.json": V04_MAN_SHA,
                                 "v0.5_manifest.json": V05_MAN_SHA},
    "written_before": ("any signal_plus_all model is trained and before any H3-type delta (signal_plus_all vs "
                       "signal_plus_physio) is computed on any fold"),
    "sealed_H3_protocol_verbatim": SEAL["H3_protocol"],
    "stage_plan": STAGE_PLAN,
    "v0.7_reused_v0.5_files": V05_REUSED_FILES,
    "v0.8_overrides": V08_OVERRIDES,
    "crash_and_rerun_rules": CRASH_AND_RERUN_RULES,
    "locked_quantities": LOCKED_QUANTITIES,
    "encoder_width_note": ENCODER_WIDTH_NOTE,
    "untrained_onehot_columns_note": UNTRAINED_COLUMNS_NOTE,
    "post_fold10_rules": POST_FOLD10_RULES,
    "bootstrap": BOOTSTRAP,
    "H3_decision": H3_DECISION,
    "reporting_order": REPORTING_ORDER,
    "exploratory_fold10_comparisons": EXPLORATORY_COMPARISONS,
    "disclosures": DISCLOSURES,
    "disclosures_fold10_exposure": DISCLOSURES_FOLD10_EXPOSURE,
    "reference_implementation": {
        "sha256_of_utf8_code": REFERENCE_CODE_SHA256,
        "usage": ("v0.9 executes this exact string (asserting sha256 of its UTF-8 bytes) and uses its functions for "
                  "every fold-10 number: fast_auc (v0.2.1, verbatim), patient grouping, BOOT_IDX, PERM_IDX, paired "
                  "bootstrap, percentile CIs, H3 verdict"),
        "code": REFERENCE_CODE,
    },
    "documents_to_timestamp": DOCUMENTS_TO_TIMESTAMP,
    "hashes_note": ("every document has two hashes: self_sha256 (canonical JSON without the key) and file sha256 "
                    "(bytes on disk, what OpenTimestamps stamps). Both are recorded."),
    "self_sha256_procedure": SELF_SHA256_PROCEDURE,
    "write_once": True,
}
PROTOCOL = plain(PROTOCOL)


def _find_nonfinite(o, path="$"):
    if isinstance(o, float) and not math.isfinite(o):
        return [path]
    if isinstance(o, dict):
        return [p for k, v in o.items() for p in _find_nonfinite(v, path + "." + str(k))]
    if isinstance(o, list):
        return [p for i, v in enumerate(o) for p in _find_nonfinite(v, "{}[{}]".format(path, i))]
    return []


assert not _find_nonfinite(PROTOCOL), "non-finite value in the protocol: {}".format(_find_nonfinite(PROTOCOL))
assert PROTOCOL["sealed_H3_protocol_verbatim"] == SEAL["H3_protocol"]
assert "OPTION A" not in json.dumps(PROTOCOL)

# --- preview (not write-once), BEFORE the protocol ---
PREVIEW_PATH = os.path.join(OUT_DIR, PREFIX + "locked_quantities_preview.json")
with open(PREVIEW_PATH, "w", encoding="utf-8") as f:
    json.dump({"note": "inspection copy, NOT write-once; the binding values are in final_run_protocol.json",
               "locked_quantities": PROTOCOL["locked_quantities"],
               "bootstrap_locked_hashes": PROTOCOL["bootstrap"]["locked_hashes"],
               "reference_implementation_sha256": REFERENCE_CODE_SHA256}, f, ensure_ascii=False, indent=2)
OUTPUT_FILES.append(PREVIEW_PATH)
print("[saved] " + PREVIEW_PATH)

if FINAL_EXISTS:
    mine = {k: v for k, v in PROTOCOL.items() if k != "timestamp"}
    disk = {k: v for k, v in FINAL_ON_DISK.items() if k not in ("timestamp", "self_sha256")}
    differing = sorted(set(mine) ^ set(disk)) + sorted(k for k in set(mine) & set(disk) if mine[k] != disk[k])
    assert not differing, "final_run_protocol.json on disk differs from this code in field(s) {} - aborting, nothing written.".format(differing)
    FINAL_SELF_SHA = FINAL_ON_DISK["self_sha256"]
    print("final_run_protocol.json: already locked (self_sha256 {}); every field except 'timestamp' is identical "
          "-> nothing written.".format(FINAL_SELF_SHA))
else:
    FINAL_SELF_SHA = write_once_json(FINAL_PROTOCOL_PATH, PROTOCOL)
    print("[saved] {} (self_sha256 {}, verified from disk)".format(FINAL_PROTOCOL_PATH, FINAL_SELF_SHA))
FINAL_FILE_SHA = sha256_of(FINAL_PROTOCOL_PATH)
OUTPUT_FILES.append(FINAL_PROTOCOL_PATH)


# =====================================================================================
# MANIFEST
# =====================================================================================
hdr("MANIFEST")
_new_modules = {m for m in _GUARDED_MODULES if m in sys.modules} - _MODULES_AT_START
assert not _new_modules, "v0.6 imported {} - this stage must stay CPU-only without them.".format(_new_modules)
seen, outputs = set(), []
for p in OUTPUT_FILES:
    if p in seen:
        continue
    seen.add(p)
    outputs.append({"file": p, "bytes": os.path.getsize(p), "sha256": sha256_of(p)})
manifest = {
    "code_version": CODE_VERSION,
    "run_started": RUN_STARTED,
    "run_finished": datetime.now().astimezone().isoformat(),
    "versions": VERSIONS,
    "input_files_sha256": INPUT_HASHES,
    "parent_documents": PROTOCOL["parent_hashes"],
    "final_run_protocol": {"path": FINAL_PROTOCOL_PATH, "self_sha256": FINAL_SELF_SHA, "file_sha256": FINAL_FILE_SHA,
                           "written_this_run": not FINAL_EXISTS},
    "reference_implementation_sha256": REFERENCE_CODE_SHA256,
    "env_regression": regression_rows,
    "bootstrap_implementation_check": BOOT_IMPL_CHECK,
    "fold10_access": FOLD10_ACCESS_LINE,
    "fold10_signal_opened": False, "fold10_label_read": False,
    "model_trained_or_evaluated": False,
    "warnings": WARNINGS,
    "outputs": outputs,
}
MANIFEST_PATH = os.path.join(OUT_DIR, PREFIX + "manifest.json")
with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2, default=_json_default)
for o in outputs:
    print("  {}  {}".format(o["sha256"], o["file"]))
print("[saved] " + MANIFEST_PATH)
print("manifest sha256:", sha256_of(MANIFEST_PATH))


# =====================================================================================
# FINAL SUMMARY
# =====================================================================================
hdr("SUMMARY - " + CODE_VERSION)
print("parent documents (self_sha256 | file sha256):")
for key in DOC_PATHS:
    print("  {:18s} {} | {}".format(key, DOC_SELF[key], DOC_FILE[key]))
print("  {:18s} {} | {}  ({})".format("final_run_protocol", FINAL_SELF_SHA, FINAL_FILE_SHA,
                                       "written this run" if not FINAL_EXISTS else "already locked, unchanged"))
print("\nencoder one-hot widths  folds 1-8 -> folds 1-9:")
for col in ACQ_COLS:
    print("  {:8s} {:3d} -> {:3d}   new levels: {}".format(col, WIDTHS18[col], WIDTHS19[col], NEW_LEVELS[col] or "-"))
print("final meta dims: {} | head first-Linear inputs: {}".format(EXPECTED_META_DIM_FINAL, HEAD_LINEAR_IN_FINAL))
print("untrained-column weight shrink: development {:.4f} | final {:.4f}".format(
    SHRINK["development_folds1to8"]["shrink_factor_exact"], SHRINK["final_folds1to9"]["shrink_factor_exact"]))
print("\nfold 10: n records {} | n patients {} | __UNKNOWN__ counts {}".format(N_F10, N_F10_PATIENTS, F10_COUNTS["n_unknown"]))
print("bootstrap / permutation locks:")
for k, v in LOCK_HASHES.items():
    print("  {:32s} {}".format(k, v))
print("reference implementation sha256 : {}".format(REFERENCE_CODE_SHA256))
print("\nwarnings ({}):".format(len(WARNINGS)))
for w in WARNINGS:
    print("  - " + w)
print("\nfiles to timestamp:")
for p in DOCUMENTS_TO_TIMESTAMP:
    print(p)
print("Next: ots stamp each file; wait for Bitcoin confirmation (ots upgrade / ots verify) before running v0.7.")
print("\nNo model was trained or evaluated in v0.6. No fold-10 signal or label was read.")



STEP 0 - INTEGRITY
pre_registration   self_sha256 5183f54569f3358a5339b16a2c3c3be49f5c7798eabe6b0bb25946119c2ba469 -> PASS
dev_protocol       self_sha256 c967ac38c8cc9382bf41fdd3785c8bbd730db196e349846b8052d856bbac5fc6 -> PASS
training_details   self_sha256 c12c33a2f649984fb1b70a46445823c46b57d1b024d87655e16661789f04b46f -> PASS
lr_selection       self_sha256 c70cbfa65e0ab75321dfef9c64714a64705c60a967685d79a878a1d40450749d -> PASS
dev_protocol       file sha256 babf101c1d69b322daef6907175fdc4c17b5a53755a68f13a998eefa0148dc2d -> PASS
training_details   file sha256 6480c191aeaed9ff1e6b1da5caa0c1e46908d453d30106a32beb7cf83678be11 -> PASS
lr_selection       file sha256 358d783b37bba6fcf61237a43817c1fd8594fe192f8ecdfca2c25b871a21415b -> PASS
pre_registration   file sha256 11dd446272654fc80d65b2fdd40259fa674bbb419a44fde84e589b6e297cde32 == v0.3_manifest.json record -> PASS
parent chain dev_protocol -> seal; training_details -> seal + dev_protocol; lr_selection -> seal + dev_protocol + train